Credit: https://www.kaggle.com/code/svanikkolli/aeroridge-engine-v2?scriptVersionId=320078869

Score original : 9.42

Run using GPU P100

In [ ]:
%%time
import numpy as np
import matplotlib.pyplot as plt

# 1. Datos simulados (parecidos a los tuyos)
y1 = np.linspace(-10, 50, 1000) 
SHIFT_VAL = abs(y1.min()) + 1

# 2. Función de Transformación
def apply_log(y_val):
    return np.log1p(np.maximum(0, y_val + SHIFT_VAL))

# 3. Función de Reversión (con y sin el clip)
def revert_log_raw(y_log):
    return np.expm1(y_log) - SHIFT_VAL

def revert_log_clipped(y_log):
    res = np.expm1(y_log) - SHIFT_VAL
    return np.clip(res, y1.min(), y1.max())

# 4. Cálculo
y_log = apply_log(y1)
y_revertida = revert_log_raw(y_log)
y_clipped = revert_log_clipped(y_log)

# 5. Visualización
plt.figure(figsize=(12, 5))
plt.plot(y1, y_revertida, label="Reversión Directa (Error)", color='red', linestyle='--')
plt.plot(y1, y_clipped, label="Reversión con Clip (Correcto)", color='green')
plt.plot(y1, y1, label="Target Real (1)", color='blue', alpha=0.3)
plt.title("Laboratorio: Impacto del Clipping en la Reversión Logarítmica")
plt.legend()
plt.show()

In [ ]:
# 6. Prueba de Estrés: Simular una predicción errónea del modelo
# El modelo predice un valor logarítmico "loco" (ej. 20), muy superior a lo normal
pred_log_loca = np.ones_like(y1) * 20 

# Reversión sin clip
error_loco = revert_log_raw(pred_log_loca) 
# Reversión con clip
seguro_loco = revert_log_clipped(pred_log_loca)

print(f"Predicción sin clip: {error_loco[0]:,.2f}")
print(f"Predicción con clip: {seguro_loco[0]:,.2f}")

In [ ]:
from pathlib import Path
import shutil
# --- LIMPIEZA TOTAL DE MODELOS ---
# Borramos la carpeta de modelos para garantizar un entrenamiento fresco
models_dir = Path("/kaggle/working/models")
if models_dir.exists():
    print(f"Borrando carpeta existente: {models_dir}")
    shutil.rmtree(models_dir)

# rogii_v34 - AeroRidge Engine v9 (proper Climber + deterministic Optuna)

**Why v34:** v33 ran 3 times with LB 9.888 / 9.897 / 9.966 - 0.078 spread.
Best LB (9.888) was *worse* than the cached baseline alone (LB 9.251).
XGBoost was not the problem.

**Root cause:** the v33 fallback hill climber dilutes the dominant model.
Reference `Climber(precision=0.001)` settles on `CB-3 (1.000) + LGB-3 (0.272)`
and reaches LB 9.251. Our v32/v33 fallback took coarse 0.5 first-improving
steps and picked up 5 models with CB-3 weight only 0.648, hurting LB by ~0.6
even though OOF was identical (10.4290 in both cases).

**Secondary cause:** Optuna `n_jobs=-1` breaks the seeded TPE sampler.
Trials complete in non-deterministic order, so alpha/tau/w_pf land at
different points each run, causing the 0.078 LB spread.

## v34 fixes (only changes from v33)
1. **Vendored proper hill climber** - multi-scale best-improvement greedy,
   step schedule 0.5 -> 0.001 with rounded-score comparison at 3 dp.
   Replaces both the Climber-package import attempt and the buggy fallback.
2. **Optuna n_jobs=1** for full determinism (cost: ~10s slower).
3. **OOF artifact filename** - v33 still wrote the previous version's name.

## Required Kaggle inputs
- `/kaggle/input/competitions/rogii-wellbore-geology-prediction` (REQUIRED)
- `/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts` (REQUIRED)
- HC notebook input no longer needed - climber is vendored.

## Runtime: ~40 min on Kaggle 2xT4 (unchanged from v33)

## Expected LB
- Proper climber picks CB-3 + LGB-3 only (matching reference): **LB 9.251**
- XGB-2 gets a small positive weight that helps OOF *and* LB: **LB 9.20-9.25**
- All 3 reruns should land within ~0.01 of each other (full determinism).
- If LB > 9.30 with proper climber, there's a third bug to chase.

## Diagnostic mode
Set `ROGII_RUN_XGB=0` to skip XGB. With the proper climber, cached
LGB+CB-only should hit LB 9.251 - that confirms the climber is the
fix and XGB is validated for inclusion in v35.


In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor, Pool
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from pathlib import Path
from numba import njit, prange
import matplotlib.pyplot as plt
import lightgbm as lgb
import multiprocessing
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
import joblib
import optuna
import time
import os
import sys
import subprocess
 
warnings.filterwarnings("ignore")

# Definición de parámetros

FLAG_CREATE_DATASET = False
FLAG_READ_1ST_TIME = True
FLAG_FEATS_REDUCED = True
N_SPLITS = 5
FLAG_TEST = False  # Cambia a False para entrenar con todo
TEST_SIZE = 400     # Número de pozos para el set de prueba
FLAG_LOG = False
if FLAG_TEST:
   N_SPLITS = 2

FLAG_OUTLIERS = False
FLAG_CREATE_DATASET = False
MODELS_LOADED = False
FLAG_MODEL = False
FLAG_READ_1ST_TIME = True
FLAG_FEATS_REDUCED = True
IS_SUBMISSION = True
N_SPLITS = 5
FLAG_TEST = True  # Cambia a False para entrenar con todo
TEST_SIZE = 5     # Número de pozos para el set de prueba
FLAG_LOG = True
if FLAG_TEST:
   N_SPLITS = 2

FEATURES = [
    #'id', 'well', 'target',                                           # Identificadores y objetivo
    'pf_ancc', 'pf_ancc_delta', 'pf_z', 'pf_z_delta', 'pf_vs_z',      # Variables pf
    'pf_vs_dense', 'spatial_vs_dense', 'spatial_ancc_d', 'spatial_knn_dist',
    'beam_mean_d', 'beam_std_d', 'beam_sm5_d', 'beam_cons_d',         # Variables beam
    'beam_stiff_d', 'beam_loose_d', 'beam_vloose_d', 'beam_vcons_d', 'beam_vs_spatial',
    'tvt_dense_d', 'tvt_dense50_d', 'tvt_densew_d', 'last_known_tvt', # Variables tvt
    'tvtF_ANCC', 'tvtF_EGFDL', 'tvtF_EGFDU', 'tvtF_ASTNU',            # Variables tvtF (corregidas)
    'tvtF50_ASTNU', 'tvtF50_ASTNL', 'tvtF50_EGFDL', 'tvtFw_EGFDL', 'tvtFw_ASTNU', 'tvtFw_ASTNL',
    'form_mean_d', 'form_std_d', 'form_rng_d',                        # Variables form
    'dtw_r20_d', 'dtw_r50_d', 'dtw_r200_d', 'dtw_vs_pf', 'dtw_vs_beam', # Variables dtw
    'dtw_ens_d', 'dtw_stoch_mean_d', 'dtw_stoch_std', 'tddtw20',
    'slp_50', 'slp_all', 'slp_b_d_50',                                # Variables slp
    'dense_dist', 'dense_ancc', 'dense_std', 'frm_rmse_ANCC', 'frm_rmse_ASTNU', # Variables dense/rmse (corregidas)
    'dx', 'dy', 'dz', 'dxy', 'frac', 'grm101', 'grs51', 'md_since', 'z' # Variables misceláneas
]

In [ ]:

NOTEBOOK_RUN_VERSION = "ROGII_v34_proper_climber_optuna_n1_2026_05_17"
_t0 = time.time()
def _dbg(msg): print(f"[{(time.time()-_t0)/60:.1f}m] {msg}", flush=True)
_dbg(f"=== {NOTEBOOK_RUN_VERSION} START ===")

# AeroRidge infra: env flags, GPU detect, CATBOOST_DEVICES
def env_flag(name, default=False):
    v = os.environ.get(name, "")
    if v == "": return default
    return v.strip().lower() not in ("0", "false", "no", "off")

_gpu_names = ""
try:
    _gpu_names = subprocess.check_output(
        "nvidia-smi --query-gpu=name --format=csv,noheader",
        shell=True, stderr=subprocess.DEVNULL, timeout=20
    ).decode(errors="ignore").strip()
except Exception:
    pass
GPU_COUNT = len([x for x in _gpu_names.splitlines() if x.strip()])
CATBOOST_DEVICES = os.environ.get(
    "ROGII_CATBOOST_DEVICES",
    ":".join(str(i) for i in range(GPU_COUNT)) if GPU_COUNT else "0",
)
RUN_XGB = env_flag("ROGII_RUN_XGB", True)
_dbg(f"GPUs: {_gpu_names!r} count={GPU_COUNT} CATBOOST_DEVICES={CATBOOST_DEVICES}")
_dbg(f"RUN_XGB={RUN_XGB}")

class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    seed = 42
    n_splits = N_SPLITS
    cv = GroupKFold(n_splits=n_splits)
    metric = root_mean_squared_error

# 2. Data loading and preprocessing

In [ ]:
SEED = 42; np.random.seed(SEED)
NCPU = min(4, multiprocessing.cpu_count())
 
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20; N_SPLITS = 5
 
BEAMS = [
    (10, 20.0, 144.0, 2, "cons"),
    (10,  8.0,  64.0, 2, "loose"),
    ( 8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0,  90.0, 5, "sm5"),
    (20,  4.0,  36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
]
 
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_INIT_V_STD = 0.02; PF_INIT_SPR = 0.5; PF_RESAMP = 0.5
PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003; PF_GR_WIN = 5; PF_GR_WT = 0.3
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005
ANCC_IR = 0.01; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001
 
DTW_RADII = (20, 50, 100, 200)
DTW_STOCH_K = 12
DTW_STOCH_TEMP = 3.0
 
 
@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i] * (1. - t) + grid[i + 1] * t
 
 
@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N + 1)
    for j in range(N): cum[j + 1] = cum[j] + w[j]
    u0 = np.random.uniform(0., 1. / N)
    np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0 + j / N
        while ci < N - 1 and cum[ci + 1] < u: ci += 1
        np2[j] = pos[ci] + rp * np.random.randn()
        na[j] = aux[ci] + rv * np.random.randn()
    return np2, na
 
 
@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30);     bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path
 
 
@njit(cache=True)
def _dtw_sakoe_chiba(query, ref, radius):
    """
    Constrained DTW with Sakoe-Chiba band.
    Returns (cost_matrix, accumulated_cost_matrix, path_i, path_j).
    Uses slanted band: diagonal from (0,0) to (N-1,M-1).
    """
    N = len(query); M = len(ref)
    INF = 1e18
    D = np.full((N, M), INF)
 
    slope = (M - 1.0) / max(N - 1.0, 1.0)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            cost = (query[i] - ref[j]) ** 2
            if i == 0 and j == 0:
                D[i, j] = cost
            elif i == 0:
                prev = D[i, j - 1]
                D[i, j] = cost + (prev if prev < INF else INF)
            elif j == 0:
                prev = D[i - 1, j]
                D[i, j] = cost + (prev if prev < INF else INF)
            else:
                a = D[i - 1, j - 1]
                b = D[i - 1, j]
                c = D[i, j - 1]
                mn = a if a < b else b
                mn = mn if mn < c else c
                D[i, j] = cost + (mn if mn < INF else INF)
 
    i = N - 1; j = M - 1
    pi = np.zeros(N + M, np.int64)
    pj = np.zeros(N + M, np.int64)
    k = 0
    while i > 0 or j > 0:
        pi[k] = i; pj[k] = j; k += 1
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
            if a <= b and a <= c:
                i -= 1; j -= 1
            elif b <= c:
                i -= 1
            else:
                j -= 1
    pi[k] = 0; pj[k] = 0; k += 1
    return D, pi[:k], pj[:k]
 
 
@njit(cache=True)
def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
    """
    Convert DTW warping path to per-query-sample TVT estimate.
    For each query index i, find the corresponding typewell index j,
    then look up tw_tvt[j].
    """
    j_for_i = np.zeros(N, np.int64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = pj[k]
    result = np.empty(N, np.float32)
    for i in range(N):
        result[i] = tw_tvt[j_for_i[i]]
    return result
 
 
@njit(cache=True)
def _dtw_path_slope(pi, pj, N, smooth_win=5):
    """
    Compute local slope dj/di along the warping path — encodes TVT rate.
    """
    j_for_i = np.zeros(N, np.float64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = float(pj[k])
 
    slope = np.zeros(N, np.float32)
    hw = smooth_win // 2
    for i in range(N):
        i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
        if i1 > i0:
            slope[i] = float((j_for_i[i1] - j_for_i[i0]) / (i1 - i0))
        else:
            slope[i] = 1.0
    return slope
 
 
@njit(cache=True)
def _dtw_stochastic_realizations(query, ref, radius, K, temperature):
    """
    Stochastic DTW: sample K realizations of the warping path by adding
    Gumbel noise to the cost matrix before traceback.
    Returns (K, N) array of typewell indices per realization.
    """
    N = len(query); M = len(ref)
    INF = 1e18
    slope = (M - 1.0) / max(N - 1.0, 1.0)
 
    D_base = np.full((N, M), INF)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            D_base[i, j] = (query[i] - ref[j]) ** 2
 
    paths = np.zeros((K, N), np.int64)
    for k in range(K):
        D = np.full((N, M), INF)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                noise = -temperature * np.log(-np.log(np.random.uniform(1e-10, 1.0)))
                cost = D_base[i, j] + noise
                if i == 0 and j == 0:
                    D[i, j] = cost
                elif i == 0:
                    prev = D[i, j - 1]
                    D[i, j] = cost + (prev if prev < INF else INF)
                elif j == 0:
                    prev = D[i - 1, j]
                    D[i, j] = cost + (prev if prev < INF else INF)
                else:
                    a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                    mn = a if a < b else b
                    mn = mn if mn < c else c
                    D[i, j] = cost + (mn if mn < INF else INF)
 
        i = N - 1; j = M - 1
        j_for_i = np.zeros(N, np.int64)
        while i > 0 or j > 0:
            j_for_i[i] = j
            if i == 0:
                j -= 1
            elif j == 0:
                i -= 1
            else:
                a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                if a <= b and a <= c:
                    i -= 1; j -= 1
                elif b <= c:
                    i -= 1
                else:
                    j -= 1
        j_for_i[0] = j
        paths[k] = j_for_i
 
    return paths
 
 
@njit(cache=True)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N,
             ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ls + IS * np.random.randn()
        rate[j] = ir + 0.01 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA * rate[j] + RN * np.random.randn()
            pos[j] += rate[j] * dm + PN * np.random.randn()
            tvt_j = pos[j] - z_v[i]
            tvt_j = max(tvt_j, vmin - 50.); tvt_j = min(tvt_j, vmin + len(gg) * step + 50.)
            pos[j] = tvt_j + z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j] - z_v[i], vmin, step)
                d = (gr_v[i] - eg) / gs
                lk = max(np.exp(-0.5 * d * d) if d * d < 600. else 0., 1e-300)
                w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1. / N
        tv = 0.
        for j in range(N): tv += w[j] * (pos[j] - z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j] * (pos[j] - z_v[i] - tv) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]
    return pts, std_
 
 
@njit(cache=True)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step,
          gs, ip, iv, beta, icpt, zsig, N,
          MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ip + 0.5 * np.random.randn()
        vel[j] = iv + 0.02 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.; pz = z_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        dzd = (z_v[i] - pz) / dm; ve = beta * dzd + icpt
        for j in range(N):
            vel[j] = MOM * vel[j] + VN * np.random.randn()
            pos[j] += vel[j] * dm + PN * np.random.randn()
            pos[j] = max(pos[j], vmin - 50.); pos[j] = min(pos[j], vmin + len(gg_p) * step + 50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step)
                dp = (gr_v[i] - ep) / gs
                lp = max(np.exp(-0.5 * dp * dp) if dp * dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step)
                    ds = (gr_sm_v[i] - es) / (gs * 1.5)
                    ls = max(np.exp(-0.5 * ds * ds) if ds * ds < 600. else 0., 1e-300)
                    lk = (1. - GR_WT) * lp + GR_WT * ls
                else:
                    lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j] - ve) / max(zsig * 2., 0.005)
            lz = max(np.exp(-0.5 * dv * dv) if dv * dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1. / N
        wm = 0.
        for j in range(N): wm += w[j] * pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j] * (pos[j] - wm) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_
 
 
def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)
 
 
def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw['TVT_input'].notna() & hw['GR'].notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values - np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))
 
 
def _nn(arr, v):
    i = int(np.searchsorted(arr, v, 'left'))
    if i >= len(arr): return len(arr) - 1
    if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v): return i - 1
    return i
 
 
def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r * 2 + 1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)
 
 
def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))
    return tw_tvt[path].astype(np.float32)
 
 
def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                        ev['GR'].values.astype(np.float64), gg, gmin, gst,
                        gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)
 
 
def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna['Z'].values); dvt = np.diff(kna['TVT_input'].values)
    dmd_k = np.diff(kna['MD'].values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2] / dmd_k[m2]; vt = dvt[m2] / dmd_k[m2]
        A = np.column_stack([vz, np.ones_like(vz)]); c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    else:
        beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2['TVT_input'].values); dmd2 = np.diff(t2['MD'].values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3] / dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw['GR'].rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                     ev['GR'].values.astype(np.float64),
                     gr_sm.loc[ev.index].values.astype(np.float64),
                     gg, gs2, gmin, gst, gs, float(kna['TVT_input'].iloc[-1]), iv,
                     beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)
 
 
def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
    """
    Multi-scale constrained DTW alignment of horizontal-well GR to typewell GR.
    For each Sakoe-Chiba radius, runs DTW and maps the warping path back to TVT space.
 
    Returns:
        dtw_tvts    : dict radius -> (N,) float32 TVT predictions
        dtw_slopes  : dict radius -> (N,) float32 local path slopes
        dtw_costs   : dict radius -> float  normalised alignment cost
        dtw_ens     : (N,) float32 cost-weighted ensemble TVT
    """
    N = len(query_gr)
    qn = (query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)
    rn = (tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)
 
    si = _nn(tw_tvt, last_tvt)
    qn_f = qn.astype(np.float64)
    rn_f = rn.astype(np.float64)
 
    dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
    inv_cost_sum = 0.0
    tvt_stack = []
 
    for r in radii:
        D, pi, pj = _dtw_sakoe_chiba(qn_f, rn_f, r)
        cost = float(D[len(qn_f) - 1, len(rn_f) - 1]) / max(len(qn_f) + len(rn_f), 1)
        tvt_pred = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt.astype(np.float32), N)
        slope = _dtw_path_slope(pi[::-1], pj[::-1], N)
        dtw_tvts[r] = tvt_pred
        dtw_slopes[r] = slope
        dtw_costs[r] = cost
        ic = 1.0 / (cost + 1e-6)
        inv_cost_sum += ic
        tvt_stack.append((tvt_pred, ic))
 
    weights = np.array([ic / inv_cost_sum for _, ic in tvt_stack], dtype=np.float32)
    tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
    dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
 
    return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens
 
 
def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt,
                       radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP):
    """
    Stochastic DTW: K noisy traceback realizations to quantify uncertainty.
    Returns (mean_tvt, std_tvt, cv_tvt) all (N,) float32.
    """
    N = len(query_gr)
    qn = ((query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)).astype(np.float64)
    rn = ((tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)).astype(np.float64)
 
    paths = _dtw_stochastic_realizations(qn, rn, radius, K, temperature)
    tvt_realiz = np.empty((K, N), dtype=np.float32)
    for k in range(K):
        for i in range(N):
            tvt_realiz[k, i] = tw_tvt[paths[k, i]]
 
    mean_tvt = tvt_realiz.mean(axis=0).astype(np.float32)
    std_tvt = tvt_realiz.std(axis=0).astype(np.float32)
    cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
    return mean_tvt, std_tvt, cv_tvt
 
 
_md = np.linspace(1, 50, 20, np.float64); _z = np.zeros(20, np.float64); _gr = np.full(20, 50., np.float64)
_gg = np.linspace(45, 55, 100, np.float64)
_pf_ancc(_md, _z, _gr, _gg, 45., 0.1, 20., 50., 0., 8, 0.998, 0.002, 0.005, 0.3, 0.1, 0.001, 0.5)
_pf_z(_md, _z, _gr, _gr, _gg, _gg, 45., 0.1, 20., 50., 0., -1., 0., 0.1, 8, 0.993, 0.005, 0.01, 0.3, 0.2, 0.003, 0.5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
_q = np.random.randn(40); _r = np.random.randn(50)
_dtw_sakoe_chiba(_q, _r, 10)
_dtw_stochastic_realizations(_q, _r, 10, 3, 2.0)
 
 
def robust_slope(x, y, w=None):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])
 
 
def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)
 
 
def seg_b_well(ktvt, kz, form_col):
    bv = ktvt + kz - form_col; n = len(bv)
    b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n - 50):])) if n >= 5 else b_full
    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02 * np.arange(n)); w /= w.sum()
    b_wls = float(np.dot(w, bv))
    return b_full, b_early, b_mid, b_late, b_wls
 
 
def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3. * scores); sw /= sw.sum(1, keepdims=True) + 1e-9
    sc_ens = (tvts * sw).sum(1).astype(np.float32)
    return out, sc_ens
 
 
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try: df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS).dropna()
            except: continue
            if len(df) == 0: continue
            row = {'wid': wid, 'x': float(df['X'].median()), 'y': float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy(); self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
 
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1),
                        (w[:, :, None] * fn).sum(1)], 1)
        try:
            coef = np.linalg.solve(A, rhs)
        except:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except: pass
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)
 
 
class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try: df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC']).dropna()
            except: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid] * len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0))
        self.tree = cKDTree(self.xy / self.scale)
 
    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32), \
               np.where(vk, dk, np.inf).min(1).astype(np.float32)
 
 
hw_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
train_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
FI = FormationPlaneKNN(train_wids, CFG.dataset_path / "train")
DI = DenseANCCImputer(train_wids, CFG.dataset_path / "train")
 
_FI = FI; _DI = DI
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
DTW_OFFS  = np.array([-20, -10, -5, -2, 0, 2, 5, 10, 20], np.float32)
 
 
def build_well(hw_path, tw_path, is_train):
    global _FI, _DI
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values('TVT')
    except:
        return None
    if is_train and 'TVT' not in hw.columns: return None
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt = tw['TVT'].to_numpy(np.float32); tw_gr = tw['GR'].to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
 
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
 
    lk = kn.iloc[-1]; last_tvt = float(lk['TVT_input'])
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
 
    bpaths = {}
    for (bs, mc, es, r, tag) in BEAMS:
        bpaths[tag] = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.
 
    ktvt = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]
    sc_cons = (sc8 + sc15 + sc25) / 3.
    sc_trust = float(np.clip(len(kn) / 200., 0., 0.6))
    hyb_ref = (1 - sc_trust) * beam_ref + sc_trust * sc_ens
 
    full_gr = gr_full.values.astype(np.float32)
    dtw_tvts_ms, dtw_slopes_ms, dtw_costs_ms, dtw_ens_ms = run_dtw_multiscale(
        full_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII
    )
 
    dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
        full_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP
    )
 
    nh = len(ev)
    ev_start = ev.index[0]
 
    def _ev_slice(arr):
        return arr[ev_start:ev_start + nh].astype(np.float32)
 
    dtw_ens_ev = _ev_slice(dtw_ens_ms)
    dtw_mean_ev = _ev_slice(dtw_mean_stoch)
    dtw_std_ev = _ev_slice(dtw_std_stoch)
    dtw_cv_ev = _ev_slice(dtw_cv_stoch)
 
    dtw_per_radius_ev = {}
    dtw_slope_ev = {}
    for r in DTW_RADII:
        dtw_per_radius_ev[r] = _ev_slice(dtw_tvts_ms[r])
        dtw_slope_ev[r] = _ev_slice(dtw_slopes_ms[r])
 
    dtw_slope_mean_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
    dtw_ens_slope_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
 
    dtw_cost_arr = np.array([dtw_costs_ms[r] for r in DTW_RADII], dtype=np.float32)
    dtw_cost_min = float(dtw_cost_arr.min())
    dtw_cost_range = float(dtw_cost_arr.max() - dtw_cost_arr.min())
 
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
    a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn['MD'].to_numpy(np.float32); kz = kn['Z'].to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr - tw_at_k) ** 2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:])
    slp_z = robust_slope(kz, ktvt)
 
    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64); xy_kn = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn['Z'].to_numpy(np.float32); z_ev = ev['Z'].to_numpy(np.float32)
 
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fw = (-z_ev + form_ev[:, fi2] + b_wls).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'tvtF_{fn}'] = tvt_f; tvt_fs[f'tvtFw_{fn}'] = tvt_fw
        tvt_fs[f'tvtF50_{fn}'] = tvt_f50
        tvt_fs[f'bw_{fn}'] = np.float32(b_full); tvt_fs[f'bww_{fn}'] = np.float32(b_wls)
        tvt_fs[f'bw50_{fn}'] = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}'] = np.float32(b_early)
        tvt_fs[f'bw_mid_{fn}'] = np.float32(b_mid)
        form_rmse[fn] = float(np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full)) ** 2)))
        form_list.append(tvt_f)
 
    fs = np.stack(form_list, 1)
    form_mean_d = (fs.mean(1) - last_tvt).astype(np.float32)
    form_std_d = fs.std(1).astype(np.float32)
    form_rng_d = (fs.max(1) - fs.min(1)).astype(np.float32)
 
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid)
    d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    b_vd = ktvt + z_kn - d_kn
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    b_d = float(np.median(b_vd))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
    tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32)
    tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn
    d_rmse = float(np.sqrt(np.mean(res_kn ** 2)))
    d_bias = float(np.mean(res_kn)); d_nb_std = float(np.mean(d_std_kn))
 
    all_sigs = [pf_use] + [p for p in bpaths.values()] + \
               [sc8, sc15, sc25, sc_ens, tvt_fs['tvtF_ANCC'], tvt_dense, dtw_ens_ev]
    sig_mat = np.stack(all_sigs, 1)
    sig_std = sig_mat.std(1).astype(np.float32)
    sig_mean = (sig_mat.mean(1) - last_tvt).astype(np.float32)
 
    gr_s = pd.Series(gr_full.values); rolls = {}
    for w in [5, 21, 51, 101]:
        r = gr_s.rolling(w, center=True, min_periods=1)
        rolls[f'grm{w}'] = r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}'] = r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        rolls[f'glag{lag}'] = gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}'] = gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1 = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2 = gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg = np.sqrt(np.maximum((gr_s ** 2).rolling(21, center=True, min_periods=1).mean(), 0.)
                     ).iloc[ev.index].values.astype(np.float32)
 
    hmd = ev['MD'].to_numpy(np.float32); md_since = hmd - float(lk['MD'])
    slp_b_all = (last_tvt + slp_all * md_since).astype(np.float32)
    slp_b_50 = (last_tvt + slp_50 * md_since).astype(np.float32)
 
    mdd = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
 
    frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
 
    def sc(v): return np.full(nh, np.float32(v), np.float32)
 
    feats = {
        'well': wid, 'id': [f'{wid}_{i}' for i in ev.index],
        'last_known_tvt': sc(last_tvt),
        'pf_ancc': pf_use, 'pf_ancc_std': std_use,
        'pf_ancc_delta': (pf_use - last_tvt).astype(np.float32),
        'pf_z': (pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta': ((pf_z - last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z': ((pf_use - pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d': (p - np.float32(last_tvt)).astype(np.float32) for t, p in bpaths.items()},
        'beam_mean_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p - last_tvt) for p in bpaths.values()], 1), 1).astype(np.float32),
        'sc8_d': (sc8 - np.float32(last_tvt)).astype(np.float32), 'sc8_sc': sc8s,
        'sc15_d': (sc15 - np.float32(last_tvt)).astype(np.float32), 'sc15_sc': sc15s,
        'sc25_d': (sc25 - np.float32(last_tvt)).astype(np.float32), 'sc25_sc': sc25s,
        'sc_cons_d': (sc_cons - np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d': (sc_ens - np.float32(last_tvt)).astype(np.float32),
        'sc_trust': sc(sc_trust), 'hyb_d': (hyb_ref - np.float32(last_tvt)).astype(np.float32),
        'sig_std': sig_std, 'sig_mean_d': sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}': sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d': form_mean_d, 'form_std_d': form_std_d, 'form_rng_d': form_rng_d,
        'spatial_ancc_d': (form_ev[:, 0] - np.float32(np.interp(last_tvt, tw_tvt, tw_gr))),
        'spatial_knn_dist': knn_d,
        'dense_ancc': d_ancc, 'dense_std': d_std, 'dense_dist': d_dist,
        'tvt_dense_d': (tvt_dense - last_tvt).astype(np.float32),
        'tvt_densew_d': (tvt_densew - last_tvt).astype(np.float32),
        'tvt_dense50_d': (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse': sc(d_rmse), 'dense_bias': sc(d_bias), 'dense_nb_std': sc(d_nb_std),
        'pf_vs_spatial': (pf_use - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense': (pf_use - tvt_dense).astype(np.float32),
        'spatial_vs_dense': (tvt_fs['tvtF_ANCC'] - tvt_dense).astype(np.float32),
        'beam_vs_spatial': (bpaths['cons'] - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam': (sc_ens - bpaths['cons']).astype(np.float32),
        'dtw_ens_d': (dtw_ens_ev - last_tvt).astype(np.float32),
        'dtw_stoch_mean_d': (dtw_mean_ev - last_tvt).astype(np.float32),
        'dtw_stoch_std': dtw_std_ev,
        'dtw_stoch_cv': dtw_cv_ev,
        'dtw_slope_mean': dtw_slope_mean_ev,
        **{f'dtw_r{r}_d': (dtw_per_radius_ev[r] - last_tvt).astype(np.float32) for r in DTW_RADII},
        **{f'dtw_slope_r{r}': dtw_slope_ev[r] for r in DTW_RADII},
        'dtw_cost_min': sc(dtw_cost_min),
        'dtw_cost_range': sc(dtw_cost_range),
        'dtw_vs_beam': (dtw_ens_ev - bpaths['cons']).astype(np.float32),
        'dtw_vs_pf': (dtw_ens_ev - pf_use).astype(np.float32),
        'dtw_vs_sc': (dtw_ens_ev - sc_ens).astype(np.float32),
        **{f'tddtw{int(o)}': hgr - np.interp(dtw_ens_ev + o, tw_tvt, tw_gr).astype(np.float32)
           for o in DTW_OFFS},
        'cal_a': sc(a_cal), 'cal_b': sc(b_cal),
        'pfx_rmse': sc(pfx_rmse), 'known_len': sc(len(kn)), 'eval_len': sc(nh),
        'slp_all': sc(slp_all), 'slp_50': sc(slp_50), 'slp_z': sc(slp_z),
        'slp_b_d_all': (slp_b_all - last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 - last_tvt).astype(np.float32),
        'ktvt_range': sc(float(np.ptp(ktvt))), 'ktvt_std': sc(float(ktvt.std())),
        'md_since': md_since, 'frac': frac, 'frac2': frac ** 2, 'sqrt_frac': np.sqrt(frac),
        'z': z_ev,
        'dx': (ev['X'] - float(lk['X'])).to_numpy(np.float32),
        'dy': (ev['Y'] - float(lk['Y'])).to_numpy(np.float32),
        'dz': (z_ev - float(lk['Z'])).astype(np.float32),
        'dxy': np.sqrt((ev['X'] - float(lk['X'])) ** 2 + (ev['Y'] - float(lk['Y'])) ** 2).to_numpy(np.float32),
        'dzdmd': dzdmd, 'dxdmd': dxdmd, 'dydmd': dydmd,
        'gr': hgr, 'gr_d1': gr_d1, 'gr_d2': gr_d2, 'gr_env': gr_env, 'gr_nrg': gr_nrg,
        'gr_vs_tw_anc': hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)),
        'gr_vs_slp_all': hgr - np.interp(slp_b_all, tw_tvt, tw_gr).astype(np.float32),
        **{f'tda{int(o)}': hgr - np.float32(np.interp(last_tvt + o, tw_tvt, tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}': hgr - np.interp(beam_ref + o, tw_tvt, tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}': hgr - np.interp(sc_ens + o, tw_tvt, tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}': hgr - np.interp(pf_use + o, tw_tvt, tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range': sc(float(np.ptp(tw_tvt))), 'tw_gr_mean': sc(float(tw_gr.mean())),
        # --- AÑADE ESTO PARA TENER LA FÍSICA ---
        'MD': hmd,                # Measured Depth de la zona de evaluación
        'Z': z_ev,                # Profundidad Vertical de la zona de evaluación
        'GR': hgr,                # Gamma Ray de la zona de evaluación
    }
    for k, v in rolls.items(): feats[k] = v
    result = pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target'] = (ev['TVT'].to_numpy(np.float32) - np.float32(last_tvt))
    return result
 
 
def build_dataset(paths, is_train, label):
    args = [(str(p), str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'), is_train)
            for p in paths
            if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()]
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=0)(
        delayed(build_well)(hp, tp, it) for hp, tp, it in args)
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

In [ ]:
%%time
if FLAG_CREATE_DATASET:
    import os
    from pathlib import Path
    from tqdm import tqdm
    from joblib import Parallel, delayed
    import pandas as pd
    
    # --- 1. Lógica de selección "Zero-Latency" ---
    # Siempre definimos test_paths primero
    test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
    
    # --- Lógica de selección (La que ya funciona) ---
    if FLAG_TEST:
        # Selección rápida basada en el sistema de archivos
        hw_paths = []
        with os.scandir(CFG.dataset_path / "train") as it:
            for entry in it:
                if entry.name.endswith('__horizontal_well.csv') and entry.is_file():
                    hw_paths.append(Path(entry.path))
                    if len(hw_paths) >= TEST_SIZE: break
    else:
        hw_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
    
    # --- Carga (Con tqdm integrado) ---
    def build_dataset(paths, is_train, label):
        args = [(str(p), str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'), is_train)
                for p in paths
                if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()]
        
        # Ejecución paralela
        res = Parallel(n_jobs=NCPU, prefer='threads')(
            delayed(build_well)(hp, tp, it) for hp, tp, it in tqdm(args, desc=f"Procesando {label}", unit="pozo")
        )
        
        parts = [r for r in res if r is not None]
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    
    # --- Ejecución ---
    train_df = build_dataset(hw_paths, is_train=True, label="train")

    test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
    test_df = build_dataset(test_paths, is_train=False, label="test")
    
    # Definición de variables
    features = [c for c in train_df.columns if c not in {'well', 'id', 'target', 'MD'}]
    y = train_df['target']
    g = train_df['well']
    X_test = test_df[features]

In [ ]:
if not FLAG_CREATE_DATASET:
    import os
    import gc

    # 1. Detección robusta de entorno
    is_submission1 = os.getenv('KAGGLE_KERNEL_RUN_TYPE') != 'Interactive'
    
    # 2. Carga Condicional Inteligente
    if not is_submission1:
        # MODO LABORATORIO
        print("🧪 Modo Laboratorio: Cargando desde caché...")
        train_df_back = pd.read_parquet('/kaggle/input/datasets/henryjavier/rogii-datasets-processed/ROGII_train_df.parquet')
        
        if FLAG_TEST:
            print(f"⚠️ TEST MODE: Filtrando a {TEST_SIZE} pozos...")
            unique_wells = train_df_back['well'].unique()[:TEST_SIZE]
            train_df = train_df_back[train_df_back['well'].isin(unique_wells)].copy()
        else:
            train_df = train_df_back.copy()
        
        del train_df_back
        gc.collect()
    else:
        # MODO SUBMISSION
        print("🚀 Modo Submission: Construyendo dataset desde fuentes originales...")
        hw_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
        train_df = build_dataset(hw_paths, is_train=True, label="train")

    # 3. Construcción del Test (Una sola vez, íntegro y sin filtros)
    print("⚙️ Procesando dataset de test completo...")
    test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
    test_df = build_dataset(test_paths, is_train=False, label="test")
    
    # 4. Preparación de Features (Centralizada)
    # Definimos features una sola vez
    features = [c for c in train_df.columns if c not in {'well', 'id', 'target', 'MD', 'Z', 'TVT', 'GR'}]
    if FLAG_FEATS_REDUCED:
        features = FEATURES
    
    # Asignación final segura
    X = train_df[features]
    y = train_df['target']
    g = train_df['well']
    
    # X_test es la matriz para los modelos; test_df mantiene su integridad para la inferencia
    X_test = test_df[features].copy()
    
    print(f"✅ Datos listos. Train shape: {train_df.shape}, Test shape: {test_df.shape}")
    gc.collect()

In [ ]:
train_df_orig = train_df.copy()

In [ ]:
%%time
import pandas as pd
import os
from tqdm import tqdm

def parchear_md(train_df, base_path):
    # Aseguramos que train_df tenga un índice de fila dentro de cada pozo
    # Esto asume que el orden de filas que sobrevivieron al dropna 
    # corresponde al orden original del CSV.
    
    # Vamos a procesar por pozo para ser seguros
    final_dfs = []
    wells = train_df['well'].unique()
    
    for well in tqdm(wells, desc="Alineando MD", unit="pozo"):
        df_well = train_df[train_df['well'] == well].copy()
        
        # Leemos el original
        file_path = os.path.join(base_path, f"{well}__horizontal_well.csv")
        df_raw = pd.read_csv(file_path, usecols=['MD'])
        
        # AQUÍ ESTÁ EL TRUCO:
        # Si eliminaste filas, el índice original de los archivos crudos 
        # se mantiene en el CSV. Si tus 225 features fueron calculadas 
        # sin cambiar el orden, puedes hacer un join.
        
        # Asignamos el índice del original como una columna temporal
        df_raw['temp_idx'] = df_raw.index
        
        # Si eliminaste filas, aquí es donde necesitamos saber qué filas quedaron.
        # Si no tienes la columna original del índice, el mejor approach es 
        # resetear el índice de tu train_df por pozo.
        df_well = df_well.reset_index(drop=True)
        
        # Hacemos el join (asumiendo que las filas sobrevivientes mantienen el orden)
        df_well['MD'] = df_raw['MD'].iloc[df_well.index].values
        
        final_dfs.append(df_well)
        
    return pd.concat(final_dfs, ignore_index=True)

# 1. Definir la ruta base siguiendo tu estructura
# Ajusta el nombre de la subcarpeta si es necesario (ej: "train")
base_path = CFG.dataset_path / "train"

# 2. Si necesitas listar los archivos para procesarlos o verificar el path:
hw_paths = sorted(base_path.glob('*__horizontal_well.csv'))

# 3. Llamar a tu función con el DataFrame existente
# Esto aplicará los parches a tu train_df usando los archivos encontrados
train_df = parchear_md(train_df, base_path)


In [ ]:
%%time
import gc
flag_outliers = 1
if flag_outliers == 1:
    # 1. Preparar datos (usando tu lógica de transformación)
    y_log = np.log1p(np.maximum(0, train_df['target'] + SHIFT_VAL))
    well_stats = pd.DataFrame({'median': train_df.groupby('well')['target'].median(),
                               'count': train_df.groupby('well').size()})
    
    # 2. Solo considerar pozos con suficiente muestra (ej. > 10 puntos)
    well_stats = well_stats[well_stats['count'] > 10]
    
    # 3. Cálculo de Z-score robusto
    mean_val = well_stats['median'].mean()
    std_val = well_stats['median'].std()
    well_stats['z_score'] = (well_stats['median'] - mean_val) / std_val
    
    # Decisión del umbral: 2 para mayor limpieza, 3 para ser conservador
    threshold = 2.0 
    outlier_wells = well_stats[abs(well_stats['z_score']) > threshold].index.tolist()
    
    # 4. Filtrado
    train_df_clean = train_df[~train_df['well'].isin(outlier_wells)].copy()
    # 1. Definir las columnas críticas que no pueden tener NaNs
    cols_criticas = ['target', 'last_known_tvt']
    
    # 2. Verificar antes de limpiar
    print(f"Filas antes de limpiar: {len(train_df)}")
    
    # 3. Eliminar filas donde target o last_known_tvt sean NaN
    # Esto asegura que 'y' y tus features clave estén alineados
    train_df_clean = train_df_clean.dropna(subset=cols_criticas).copy()
    # 3. Reseteamos el índice para que sea 0, 1, 2, ..., N
    train_df_clean = train_df_clean.reset_index(drop=True)
    print(f"🚫 Pozos eliminados (Umbral {threshold}): {outlier_wells}")
    print(f"✅ Filas restantes: {len(train_df_clean)} de {len(train_df)}")
    train_df = train_df_clean.copy()
    del train_df_clean
    gc.collect()

In [ ]:
train_df_orig.shape

In [ ]:
train_df.shape, test_df.shape, X.shape, y.shape, X_test.shape

In [ ]:
#train_df.shape, test_df.shape, X.shape, y.shape, X_test.shape

In [ ]:
%%time
# Escribir zipeado
#train_df.to_csv("/kaggle/working/ROGII_train_df.gz", index=False, compression='gzip')

In [ ]:
%%time
# Escribir zipeado
#test_df.to_csv("/kaggle/working/ROGII_test_df.gz", index=False, compression='gzip')

In [ ]:
%%time
if FLAG_CREATE_DATASET:
    # Guarda manteniendo la integridad total (float64) pero usando ZSTD
    train_df.to_parquet("/kaggle/working/ROGII_train_df.parquet", compression='zstd')
    test_df.to_parquet("/kaggle/working/ROGII_test_df.parquet", compression='zstd')
    print("Datos guardados en formato Parquet.")

In [ ]:
%%time
# 2. Si realmente necesitas comprimirlos más para subirlos o moverlos, 
# entonces comprimes el archivo binario resultante:
#import shutil
#shutil.make_archive('/kaggle/working/ROGII_datasets', 'zip', '/kaggle/working/', base_dir='ROGII_train_df.parquet')

In [ ]:
%%time
if FLAG_CREATE_DATASET:
    import json
    import os
    
    # REEMPLAZA ESTO POR TU NOMBRE DE USUARIO REAL
    mi_usuario = "henryjavier" 
    dataset_name = "rogii-datasets-processed"
    
    metadata = {
      "title": dataset_name,
      "id": f"{mi_usuario}/{dataset_name}",
      "licenses": [{"name": "CC0-1.0"}]
    }
    
    with open('/kaggle/working/dataset-metadata.json', 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"Metadata actualizada para: {mi_usuario}/{dataset_name}")

In [ ]:
%%time
if FLAG_CREATE_DATASET:
    !kaggle datasets create -p /kaggle/working/

In [ ]:
#!rm /kaggle/working/ROGII_test_df.parquet

In [ ]:
#!rm /kaggle/working/ROGII_train_df.parquet

In [ ]:
def create_robust_features(df):
    df_new = df.copy()
    
    # Ejemplo 1: Gradientes (¿cómo cambia la densidad respecto a la profundidad Z?)
    # Esto captura la física del pozo sin mirar el target
    df_new['grad_density'] = df_new['density'].diff() / (df_new['z'].diff() + 1e-6)
    
    # Ejemplo 2: Rolling Stats (Contexto del entorno)
    # Media y desviación estándar de 10 filas hacia arriba (vecindad)
    df_new['gr_rolling_mean'] = df_new.groupby('well')['gr'].transform(lambda x: x.rolling(10, min_periods=1).mean())
    df_new['gr_rolling_std'] = df_new.groupby('well')['gr'].transform(lambda x: x.rolling(10, min_periods=1).std())
    
    # Ejemplo 3: Ratios Físicos (ej. Litología aproximada)
    # Relación entre resistividad y densidad
    df_new['res_den_ratio'] = df_new['resistivity'] / (df_new['density'] + 1e-6)
    
    return df_new

# Filtra las columnas que no sean las que quieres borrar
cols_disponibles = [c for c in train_df.columns if 'tvt' not in c.lower() and 'frm_rmse' not in c.lower()]

print("📌 Tus columnas disponibles (sin las sospechosas de fuga):")
print(sorted(cols_disponibles))

In [ ]:
len(FEATURES)

In [ ]:
# 1. Definir las columnas críticas que no pueden tener NaNs
cols_criticas = ['target', 'last_known_tvt']

# 2. Verificar antes de limpiar
print(f"Filas antes de limpiar: {len(train_df)}")

# 3. Eliminar filas donde target o last_known_tvt sean NaN
# Esto asegura que 'y' y tus features clave estén alineados
train_df_clean = train_df.dropna(subset=cols_criticas).copy()

# 4. Sincronizar 'y' y resetear índices para evitar errores de alineación en TimeSeriesSplit
train_df_clean = train_df_clean.reset_index(drop=True)


In [ ]:
train_df_1 = train_df.copy()

In [ ]:
FLAG_LOG

In [ ]:
train_df.shape, X.shape, y.shape, X_test.shape

In [ ]:
#train_df.shape, X.shape, y.shape, X_test.shape

In [ ]:
#train_df = train_df_orig.copy()

In [ ]:
%%time
# --- 3. PREPARACIÓN DE FEATURES PARA EL MODELO ---
features = [c for c in train_df.columns if c not in {'well', 'id', 'target', 'MD'}]
if FLAG_FEATS_REDUCED:
    features = FEATURES
X = train_df[features]
y = train_df['target']
g = train_df['well']
X_test = test_df[features]

print(f"✅ Datos listos para el entrenamiento. Train shape: {train_df.shape}")

In [ ]:
train_df.shape, len(features)

# 3. Training

In [ ]:
%%time
import torch
# Detección automática
IS_GPU_AVAILABLE = torch.cuda.is_available()

# Mapeo automático
DEVICE = "gpu" if IS_GPU_AVAILABLE else "cpu"

print(f"--- Hardware detectado: {'GPU' if IS_GPU_AVAILABLE else 'CPU'} ---")

lgb_params_base = dict(
    boosting_type="gbdt", num_leaves=255, min_child_samples=15,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=3.0, reg_alpha=0.05, objective="regression",
    verbose=-1, n_jobs=-1, device_type=DEVICE, gpu_use_dp=False, max_bin=255,
)
 
lgb_params = [
    dict(learning_rate=0.025, n_estimators=8000, seed=42, **lgb_params_base),
    dict(learning_rate=0.020, n_estimators=8000, seed=7, **lgb_params_base),
    dict(learning_rate=0.030, n_estimators=8000, seed=123, **lgb_params_base),
]

# Mantenemos las constantes base, pero ajustamos para mayor capacidad de generalización
lgb_params_base = dict(
    boosting_type="gbdt", 
    objective="regression",
    verbose=-1, 
    n_jobs=-1, 
    device_type=DEVICE, 
    gpu_use_dp=False, 
    max_bin=255
)

# Definimos los modelos con diversidad explícita
lgb_params = [
    # Modelo 1: Conservador y profundo (Especialista en detalle)
    dict(learning_rate=0.01, n_estimators=8000, num_leaves=64, min_child_samples=30, 
         subsample=0.7, colsample_bytree=0.7, reg_lambda=5.0, reg_alpha=0.1, seed=42, **lgb_params_base),
    
    # Modelo 2: Rápido y agresivo (Especialista en tendencia)
    dict(learning_rate=0.05, n_estimators=8000, num_leaves=255, min_child_samples=10, 
         subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, reg_alpha=0.01, seed=7, **lgb_params_base),
    
    # Modelo 3: Balanceado (Especialista en generalización)
    dict(learning_rate=0.025, n_estimators=8000, num_leaves=128, min_child_samples=20, 
         subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, seed=123, **lgb_params_base),
]
 
cb_params_base = dict(
    iterations=8000, depth=7, l2_leaf_reg=2.0,
    min_data_in_leaf=15, border_count=254,
    loss_function="RMSE", task_type=DEVICE, devices=CATBOOST_DEVICES,
    od_type="Iter", od_wait=300, verbose=0,
)

 
cb_params = [
    dict(learning_rate=0.025, random_seed=42, **cb_params_base),
    dict(learning_rate=0.020, random_seed=7, **cb_params_base),
    dict(learning_rate=0.030, random_seed=123, **cb_params_base),
]

# NEW v33: XGBoost - third GBDT family for ensemble diversity.
# tree_method="hist" + device="cuda" runs on GPU 1 (CB uses both via CATBOOST_DEVICES).
xgb_params_base = dict(
    booster="gbtree",
    max_depth=8,
    min_child_weight=15.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3.0,
    reg_alpha=0.05,
    objective="reg:squarederror",
    eval_metric="rmse",
    #device="cuda",
    device=DEVICE,
    tree_method="hist",
    max_bin=255,
    verbosity=0,
)
xgb_params = [
    dict(learning_rate=0.025, n_estimators=8000, random_state=42, **xgb_params_base),
    dict(learning_rate=0.020, n_estimators=8000, random_state=7, **xgb_params_base),
    dict(learning_rate=0.030, n_estimators=8000, random_state=123, **xgb_params_base),
]

In [ ]:
#y.shape, X.shape

## 3.1 LightGBM

In [ ]:
%%time

import shutil
from pathlib import Path
from sklearn.model_selection import KFold
import os
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error


oof_preds = {}
test_preds = {}

overall_scores = {}
fold_scores = {}


# 2. Configuración de constantes
SHIFT_VAL = abs(y.min()) + 1 
def apply_log(y_val):
    if not FLAG_LOG: return y_val
    return np.log1p(np.maximum(0, y_val + SHIFT_VAL))

def revert_log_clipped(y_log):
    if not FLAG_LOG: return y_log
    res = np.expm1(y_log) - SHIFT_VAL
    return np.clip(res, y.min(), y.max())


# 1. Limpieza de directorio
models_dir = Path("/kaggle/working/models")
if models_dir.exists() and models_dir.is_dir():
    shutil.rmtree(models_dir)
    print(f"✅ Directorio eliminado.")

if not FLAG_MODEL:
    
    # 1. Definición de la función de entrenamiento (siempre disponible)
    def train_lightgbm(params_input, name, X, y, X_test):
        params = params_input.copy()
        params.setdefault('objective', 'huber')
        params.setdefault('huber_delta', 1.0)
        params.setdefault('max_bin', 63)
        
        n_est = params.pop('n_estimators', 10000)
        y_log = apply_log(y)
        path = models_dir / name
        
        oof_preds_log = np.full(len(X), np.nan, dtype=np.float32)
        test_preds_log_sum = np.zeros(len(X_test), dtype=np.float32)
        models = []
        
        X_df = X if isinstance(X, pd.DataFrame) else X.to_frame()
        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
        
        print(f"--- Iniciando K-Fold training para {name} ---")
        
        for fold_idx, (train_idx, valid_idx) in enumerate(kf.split(X_df)):
            x_train, x_valid = X_df.iloc[train_idx], X_df.iloc[valid_idx]
            y_train, y_valid = y_log.iloc[train_idx], y_log.iloc[valid_idx]
            
            d_train = lgb.Dataset(x_train, label=y_train)
            d_valid = lgb.Dataset(x_valid, label=y_valid, reference=d_train)
            
            m = lgb.train(
                params, d_train, 
                valid_sets=[d_train, d_valid], 
                num_boost_round=n_est,
                callbacks=[lgb.early_stopping(250, verbose=False), lgb.log_evaluation(1000)]
            )
            
            models.append(m)
            preds_fold_log = m.predict(x_valid, num_iteration=m.best_iteration)
            oof_preds_log[valid_idx] = preds_fold_log.astype(np.float32)
            test_preds_log_sum += m.predict(X_test, num_iteration=m.best_iteration).astype(np.float32)
            
            # --- CÁLCULO DE RMSE POR FOLD (Asegúrate de tener esto en tu código) ---
            fold_score = root_mean_squared_error(revert_log_clipped(y_valid), revert_log_clipped(preds_fold_log))
            print(f"Fold {fold_idx} RMSE: {fold_score:.4f}")
            
        # Finalización
        os.makedirs(path, exist_ok=True)
        joblib.dump(models, path / "models.pkl")
        
        final_test_preds = revert_log_clipped(test_preds_log_sum / N_SPLITS)
        overall_score = root_mean_squared_error(revert_log_clipped(y_log), revert_log_clipped(oof_preds_log))
        print(f"✅ Finalizado {name} | Overall RMSE: {overall_score:.4f}")
        
        return oof_preds_log, final_test_preds, overall_score
    
    # 2. Lógica de Ejecución
    models_dir = Path("/kaggle/working/models")
    
    # --- LÓGICA DE EJECUCIÓN CORREGIDA ---
    if not FLAG_MODEL:
        print("🚀 MODO ENTRENAMIENTO: Ejecutando y cargando en ALL_MODELS...")
        
        if models_dir.exists(): shutil.rmtree(models_dir)
        
        oof_preds = {}
        test_preds = {}
        overall_scores = {}
        ALL_MODELS = [] # <--- Aseguramos que inicie limpia
        
        for i in range(3):
            print(f"--- Iniciando entrenamiento de lightgbm-{i + 1} ---")
            _oof, _test, _score = train_lightgbm(lgb_params[i], f"lightgbm-{i + 1}", X, y, X_test)
            
            # AQUÍ ESTÁ EL CAMBIO: Cargamos el objeto en memoria
            # Como train_lightgbm guarda una lista de modelos por fold en disco, 
            # cargamos lo que acabamos de guardar para tenerlo en la variable global:
            models_path = models_dir / f"lightgbm-{i + 1}" / "models.pkl"
            models_list = joblib.load(models_path)
            ALL_MODELS.extend(models_list) 
            
            oof_preds[f"lightgbm-{i + 1}"] = _oof
            test_preds[f"lightgbm-{i + 1}"] = _test
            overall_scores[f"lightgbm-{i + 1}"] = _score
            
        print(f"✅ Se han cargado {len(ALL_MODELS)} modelos en ALL_MODELS.")
    else:
        print("🔍 MODO INFERENCIA: FLAG_MODEL es True. ALL_MODELS se cargará desde el Dataset.")

In [ ]:
# 1. Definimos la métrica real (sin logaritmos)
def get_real_rmse(y_true, y_pred_log):
    # Revertimos el logaritmo a la escala original
    y_true_real = y_true # y ya es la escala original en tu variable global
    y_pred_real = revert_log_clipped(y_pred_log)
    return root_mean_squared_error(y_true_real, y_pred_real)

if not FLAG_MODEL:

    # 2. Análisis por modelo (esto te dirá cuál es el mejor)
    print("--- Análisis de desempeño en Espacio Real ---")
    for model_name, preds_log in oof_preds.items():
        # Filtramos los NaNs de los folds (si existieran)
        valid_mask = ~np.isnan(preds_log)
        
        rmse_real = get_real_rmse(y[valid_mask], preds_log[valid_mask])
        print(f"Modelo {model_name} | RMSE Real: {rmse_real:.4f}")
    
    # 3. Comparativa visual (opcional pero recomendada)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 5))
    plt.scatter(y, revert_log_clipped(list(oof_preds.values())[0]), alpha=0.5, label='Predicciones')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Ideal (x=y)')
    plt.xlabel("Valor Real")
    plt.ylabel("Predicción Real")
    plt.title("¿Qué tan cerca estamos de la realidad?")
    plt.legend()
    plt.show()

In [ ]:
%%time
import os
from pathlib import Path
import lightgbm as lgb

# --- INICIALIZACIÓN DE SEGURIDAD ---
if 'ALL_MODELS' not in globals() or ALL_MODELS is None:
    ALL_MODELS = []

def cargar_modelos_si_necesario():
    global ALL_MODELS 
    
    # NUEVO: Bloqueo de seguridad. Si estamos entrenando (FLAG_MODEL es False),
    # no queremos que se ejecute la carga automática.
    if not FLAG_MODEL:
        print("⚠️ FLAG_MODEL es False: Saltando carga automática de modelos (Entrenamiento en curso).")
        return

    # Si estamos en modo inferencia, procedemos normalmente
    is_submission = os.environ.get('KAGGLE_IS_COMPETITION_RERUN') == '1'
    
    if ALL_MODELS is None: 
        ALL_MODELS = []
    
    if not is_submission and len(ALL_MODELS) == 0:
        print("🔍 Modo Local: Cargando modelos desde disco...")
        models_dir = Path("/kaggle/input/datasets/henryjavier/modelos-20-pozos")
        
        nuevos_modelos = [lgb.Booster(model_file=str(models_dir / f"model_{i}.txt")) 
                          for i in range(15) if (models_dir / f"model_{i}.txt").exists()]
        
        if len(nuevos_modelos) > 0:
            ALL_MODELS = nuevos_modelos
            print(f"✅ {len(ALL_MODELS)} modelos cargados.")
        else:
            print("⚠️ No se encontraron archivos de modelo en la ruta.")
            
    elif is_submission:
        print("🚀 Modo Submission: Usando modelos pre-cargados.")
    else:
        print(f"ℹ️ ALL_MODELS ya tiene {len(ALL_MODELS)} modelos.")

# --- EJECUCIÓN ---
# Ahora la ejecución es limpia y directa
cargar_modelos_si_necesario()

if FLAG_MODEL:
    # Tus otros imports y lógica de validación...
    # (El resto de tu código se mantiene igual, pero protegido por la función anterior)
    pass

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
     
    # --- CELDA DE REPORTE: TABLA DETALLADA POR POZO ---
    import pandas as pd
    import numpy as np
    
    # 1. Recuperamos los datos del subset utilizado
    df_detallado = train_df.loc[X_test_temp.index].copy()
    df_detallado['target'] = y_test_temp.values
    df_detallado['prediccion'] = preds_temp_real
    
    # 2. Agrupamos por pozo
    # Usamos .agg para evitar advertencias de versiones recientes de pandas
    tabla_pozos = df_detallado.groupby('well').apply(lambda x: pd.Series({
        'Target_Prom': x['target'].mean(),
        'Pred_Prom': x['prediccion'].mean(),
        'Diferencia_Prom': (x['target'] - x['prediccion']).mean(),
        'RMSE_Pozo': np.sqrt(np.mean((x['target'] - x['prediccion'])**2))
    }), include_groups=False)
    
    # 3. Calculamos el total global para agregarlo como fila final
    fila_total = pd.DataFrame({
        'Target_Prom': [df_detallado['target'].mean()],
        'Pred_Prom': [df_detallado['prediccion'].mean()],
        'Diferencia_Prom': [(df_detallado['target'] - df_detallado['prediccion']).mean()],
        'RMSE_Pozo': [rmse_holdout]
    }, index=['TOTAL_GLOBAL'])
    
    # 4. Unimos todo
    tabla_final = pd.concat([tabla_pozos, fila_total])
    
    # 5. Formateo y visualización
    pd.options.display.float_format = '{:.4f}'.format
    print("--- TABLA DETALLADA: DESEMPEÑO POR POZO ---")
    display(tabla_final)

In [ ]:
%%time
if FLAG_MODEL:
    # --- CELDA DE REPORTE FINAL (AUTÓNOMA, CON TIMER Y PROGRESO) ---
    import joblib
    import numpy as np
    import time
    from pathlib import Path
    from tqdm import tqdm
    
    # 1. Definir rutas y configuración
    models_dir = Path("models") 
    model_names = ["lightgbm-1", "lightgbm-2", "lightgbm-3"]
    
    # 2. Re-crear X_test_temp si no está en memoria
    if 'X_test_temp' not in globals():
        print("⚠️ X_test_temp no estaba en memoria, recreando desde train_df...")
        unique_wells = train_df['well'].unique()
        test_temp_wells = unique_wells[-20:] 
        mask_temp = train_df['well'].isin(test_temp_wells)
        X_test_temp = train_df.loc[mask_temp, features]
        y_test_temp = train_df.loc[mask_temp, 'target']
    
    # 3. Carga de modelos
    ALL_MODELS = []
    print(f"📦 Cargando modelos desde {models_dir}...")
    for name in model_names:
        path = models_dir / name
        if path.exists():
            # Asumiendo que joblib.load devuelve una lista de modelos
            models = joblib.load(path / "models.pkl")
            if isinstance(models, list):
                ALL_MODELS.extend(models)
            else:
                ALL_MODELS.append(models)
            print(f"   ✓ Modelos de {name} cargados.")
        else:
            print(f"   ⚠️ No se encontró {name}, saltando.")
    
    # 4. Inferencia con barra de progreso y timer
    print(f"\n🚀 Iniciando inferencia en {len(ALL_MODELS)} modelos...")
    start_time = time.time()
    
    preds_list = []
    for m in tqdm(all_models, desc="Inferencia en curso"):
        preds_list.append(m.predict(X_test_temp))
    
    preds_temp_log = np.mean(preds_list, axis=0)
    preds_temp_real = revert_log_clipped(preds_temp_log)
    
    end_time = time.time()
    print(f"✅ Inferencia completada en {end_time - start_time:.2f} segundos.")
    
    # 5. Cálculo de métricas
    rmse_holdout = root_mean_squared_error(y_test_temp, preds_temp_real)
    mae_holdout = np.abs(y_test_temp - preds_temp_real).mean()
    
    print(f"\n📊 Resultados sobre {len(y_test_temp)} registros (Test_temp):")
    print(f"   RMSE Real: {rmse_holdout:.4f}")
    print(f"   MAE Real:  {mae_holdout:.4f}")
    
    # Diagnóstico simple
    if rmse_holdout < 7.5:
        print("\n✅ ¡Excelente! El modelo generaliza bien por debajo del benchmark.")
    else:
        print(f"\n⚠️ El error es de {rmse_holdout:.4f}. Considera ajustar el modelo.")

In [ ]:
'''
import joblib
from pathlib import Path

def recuperar_ALL_MODELS_desde_disco():
    global ALL_MODELS
    ALL_MODELS = [] # Limpiamos para evitar duplicados
    
    models_dir = Path("./models")
    
    if not models_dir.exists():
        print("❌ El directorio ./models no existe.")
        return

    # Iteramos sobre las carpetas creadas (lightgbm-1, lightgbm-2, lightgbm-3)
    # Ordenamos para asegurar que el orden sea consistente
    for i in range(1, 4):
        model_path = models_dir / f"lightgbm-{i}" / "models.pkl"
        
        if model_path.exists():
            print(f"📦 Cargando modelos desde {model_path}...")
            # joblib.load devuelve la lista de modelos de los folds
            modelos_fold = joblib.load(model_path)
            ALL_MODELS.extend(modelos_fold)
        else:
            print(f"⚠️ No se encontró {model_path}")
            
    print(f"✅ Recuperación finalizada. Total modelos en ALL_MODELS: {len(ALL_MODELS)}")

# Ejecución
recuperar_ALL_MODELS_desde_disco()
'''

In [ ]:
%%time
if not FLAG_MODEL:
    from sklearn.metrics import classification_report, confusion_matrix

    # 1. Preparar los datos de validación (Tus 20 pozos)
    unique_wells = train_df['well'].unique()
    test_temp_wells = unique_wells[-20:] 
    mask_temp = train_df['well'].isin(test_temp_wells)
    
    X_test_temp = train_df.loc[mask_temp, features]
    y_test_temp = train_df.loc[mask_temp, 'target']

    # 2. GENERAR PREDICCIONES (Aquí faltaba este paso)
    # Debemos pasar los datos por ALL_MODELS para crear preds_temp_real
    preds_acumulado = np.zeros(len(X_test_temp))
    for m in ALL_MODELS:
        preds_acumulado += m.predict(X_test_temp)
    
    # Asegúrate de usar tu función de reversión logarítmica
    preds_temp_real = revert_log_clipped(preds_acumulado / len(ALL_MODELS))

    # 3. Definir el umbral y clasificar
    umbral = np.median(y_test_temp)
    print(f"🎯 Umbral aplicado: {umbral:.4f}")
    
    y_true_bin = (y_test_temp > umbral).astype(int)
    y_pred_bin = (preds_temp_real > umbral).astype(int)
    
    # 4. Reporte de Clasificación
    print("\n=== REPORTE DE CLASIFICACIÓN (Umbral Mediana) ===")
    print(classification_report(y_true_bin, y_pred_bin, target_names=['Bajo (<=Med)', 'Alto (>Med)']))
    
    # 5. Matriz de Confusión
    cm = confusion_matrix(y_true_bin, y_pred_bin)
    df_cm = pd.DataFrame(cm, index=['Real_Bajo', 'Real_Alto'], columns=['Pred_Bajo', 'Pred_Alto'])
    
    print("\n--- MATRIZ DE CONFUSIÓN (Tabular) ---")
    display(df_cm)
    
    accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
    print(f"\n✅ Precisión Global en Clasificación: {accuracy:.2%}")

In [ ]:
if not FLAG_MODEL and IS_SUBMISSION:
    # En lugar de matriz de confusión, usa un histograma de errores
    error_dist = preds_temp_real - y_test_temp
    plt.hist(error_dist, bins=50)
    plt.title("Distribución del Error (¿Es ruido centrado en cero?)")
    plt.show()

In [ ]:
%%time
if not FLAG_MODEL and IS_SUBMISSION:
    # 1. Asegúrate de incluir 'well' en el dataframe de resultados
    # Si df_detallado viene de un filtrado, asegúrate de mantener 'well'
    # Ejemplo: df_detallado = train_df.loc[mask_temp, ['well', 'target', 'prediccion']]

    df_detallado = pd.DataFrame({
    'well': train_df.loc[mask_temp, 'well'].values, 
    'target': y_test_temp.values,
    'prediccion': preds_temp_real
    })
    
    # 2. Simplifica el cálculo usando columnas existentes (es más rápido que lambda)
    df_detallado['error'] = df_detallado['target'] - df_detallado['prediccion']
    df_detallado['error_sq'] = df_detallado['error']**2
    
    # 3. Calculamos las estadísticas
    tabla_pozos = df_detallado.groupby('well').agg(
        Target_Prom=('target', 'mean'),
        Pred_Prom=('prediccion', 'mean'),
        Diferencia_Prom=('error', 'mean'),
        RMSE_Pozo=('error_sq', lambda x: np.sqrt(x.mean()))
    )
    
    # 4. Cálculo global (usando las mismas columnas ya calculadas)
    fila_total = pd.DataFrame({
        'Target_Prom': [df_detallado['target'].mean()],
        'Pred_Prom': [df_detallado['prediccion'].mean()],
        'Diferencia_Prom': [df_detallado['error'].mean()],
        'RMSE_Pozo': [np.sqrt(df_detallado['error_sq'].mean())]
    }, index=['TOTAL_GLOBAL'])
    
    # 5. Concatenar
    tabla_final = pd.concat([tabla_pozos, fila_total])
    
    pd.options.display.float_format = '{:.4f}'.format
    display(tabla_final)

In [ ]:
%%time
if not FLAG_MODEL and IS_SUBMISSION:
    # 1. Calculamos las estadísticas por pozo
    tabla_pozos = df_detallado.groupby('well').agg(
        Target_Prom=('target', 'mean'),
        Pred_Prom=('prediccion', 'mean'),
        Diferencia_Prom=('target', lambda x: (x - df_detallado.loc[x.index, 'prediccion']).mean()),
        RMSE_Pozo=('target', lambda x: np.sqrt(np.mean((x - df_detallado.loc[x.index, 'prediccion'])**2)))
    )
    
    # 2. Calculamos los valores globales reales (Promedios totales, no sumas)
    rmse_global_real = np.sqrt(np.mean((df_detallado['target'] - df_detallado['prediccion'])**2))
    target_global_mean = df_detallado['target'].mean()
    pred_global_mean = df_detallado['prediccion'].mean()
    dif_global_mean = (df_detallado['target'] - df_detallado['prediccion']).mean()
    
    # 3. Creamos una fila de totales usando un DataFrame con los mismos nombres de columnas
    fila_total = pd.DataFrame({
        'Target_Prom': [target_global_mean],
        'Pred_Prom': [pred_global_mean],
        'Diferencia_Prom': [dif_global_mean],
        'RMSE_Pozo': [rmse_global_real]
    }, index=['TOTAL_GLOBAL']) # Cambiamos el índice para que no se confunda
    
    # 4. Concatenamos
    tabla_final = pd.concat([tabla_pozos, fila_total])
    
    pd.options.display.float_format = '{:.4f}'.format
    print("--- TABLA DETALLADA: DESEMPEÑO POR POZO ---")
    display(tabla_final)
    
    # Verificación independiente
    print(f"RMSE Global Real calculado manualmente: {np.sqrt(np.mean((df_detallado['target'] - df_detallado['prediccion'])**2)):.4f}")

In [ ]:
%%time
if not FLAG_MODEL and IS_SUBMISSION:
    from sklearn.metrics import roc_curve
    
    # 1. Calcular FPR, TPR y umbrales posibles
    fpr, tpr, thresholds = roc_curve(y_true_bin, preds_temp_real)
    
    # 2. Calcular el Índice de Youden para cada umbral
    j_scores = tpr - fpr
    ix = np.argmax(j_scores) # Índice del umbral óptimo
    umbral_optimo = thresholds[ix]
    
    print(f"🎯 El umbral óptimo (Youden) es: {umbral_optimo:.4f}")

In [ ]:
if not FLAG_MODEL and IS_SUBMISSION:
    # 1. Definir el Umbral Óptimo y un buffer basado en el RMSE (la incertidumbre natural)
    umbral_real = umbral_optimo
    buffer_real = 0.0506 # Tu RMSE global real, que representa la incertidumbre del modelo
    
    # 2. Clasificación lógica (Centrada en el umbral real)
    def clasificar_con_modelo(valor, umbral, buf):
        if valor < (umbral - buf): return "Bajo"
        if valor > (umbral + buf): return "Alto"
        return "Zona_de_Duda_Modelado"
    
    # 3. Aplicar al reporte
    df_reporte_final = df_detallado.copy()
    df_reporte_final['real_cat'] = [clasificar_con_modelo(v, umbral_real, buffer_real) for v in y_test_temp]
    df_reporte_final['pred_cat'] = [clasificar_con_modelo(v, umbral_real, buffer_real) for v in preds_temp_real]
    
    # 1. Filtramos solo lo que no es duda para un reporte limpio
    df_reporte_limpio = df_reporte_final[df_reporte_final['real_cat'] != "Zona_de_Duda_Modelado"].copy()
    
    # 2. Generamos el reporte usando solo las categorías reales
    print("=== REPORTE FINAL: AUDITORÍA DE CLASIFICACIÓN (ZONA SEGURA) ===")
    print(classification_report(df_reporte_limpio['real_cat'], df_reporte_limpio['pred_cat']))
    
    # 3. Identificación de los 'Rebeldes' (los que cayeron en duda)
    df_duda = df_reporte_final[df_reporte_final['real_cat'] == "Zona_de_Duda_Modelado"]
    print(f"\n--- ANÁLISIS DE LOS {len(df_duda)} CASOS EN ZONA DE DUDA ---")
    print(df_duda.groupby('well').size().sort_values(ascending=False).head(5))

In [ ]:
if not FLAG_MODEL and IS_SUBMISSION:
    import joblib
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from pathlib import Path
    
    # Lista de rutas a las carpetas donde guardaste tus modelos
    model_dirs = [
        "/kaggle/working/models/lightgbm-1",
        "/kaggle/working/models/lightgbm-2",
        "/kaggle/working/models/lightgbm-3"
    ]
    
    # Acumulador de importancia
    all_importances = []
    
    for dir_path in model_dirs:
        # 1. Cargar la lista de modelos (Boosters) desde el archivo pkl
        models_list = joblib.load(Path(dir_path) / "models.pkl")
        
        # 2. Calcular la importancia de cada modelo en la lista
        for m in models_list:
            imp = pd.DataFrame({
                'feature': m.feature_name(),
                'gain': m.feature_importance(importance_type='gain')
            })
            all_importances.append(imp)
    
    # 3. Consolidar y promediar
    combined_imp = pd.concat(all_importances)
    avg_imp = combined_imp.groupby('feature')['gain'].mean().sort_values(ascending=False)
    
    # 4. Visualizar
    plt.figure(figsize=(10, 8))
    avg_imp.head(20).plot(kind='barh', color='skyblue')
    plt.title('Top 20 Features por Ganancia (Promedio de todos los modelos)')
    plt.xlabel('Ganancia Promedio')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # 5. Imprimir Top 10
    print("🏆 Top 10 variables más importantes:")
    print(avg_imp.head(10))

In [ ]:
%%time
if not FLAG_MODEL and IS_SUBMISSION:
    import joblib
    import pandas as pd
    import numpy as np
    from pathlib import Path
    
    # --- CONFIGURACIÓN ---
    model_dirs = [
        "/kaggle/working/models/lightgbm-1",
        "/kaggle/working/models/lightgbm-2",
        "/kaggle/working/models/lightgbm-3"
    ]
    
    # --- 1. RECOLECCIÓN DE IMPORTANCIAS ---
    all_importances = []
    for dir_path in model_dirs:
        # Cargamos la lista de modelos (boosters)
        models_list = joblib.load(Path(dir_path) / "models.pkl")
        for m in models_list:
            imp = pd.DataFrame({
                'feature': m.feature_name(),
                'gain': m.feature_importance(importance_type='gain')
            })
            all_importances.append(imp)
    
    # --- 2. CÁLCULO Y AGRUPACIÓN ---
    # Agrupamos por nombre de feature y promediamos la ganancia
    combined_imp = pd.concat(all_importances).groupby('feature')['gain'].mean().reset_index()
    
    # --- 3. PROCESAMIENTO PARA EL REPORTE (PARETO) ---
    # Ordenamos de mayor a menor ganancia PRIMERO
    final_report = combined_imp.sort_values(by='gain', ascending=False).reset_index(drop=True)
    
    # Calculamos porcentajes sobre el total
    total_gain = final_report['gain'].sum()
    final_report['pct'] = (final_report['gain'] / total_gain) * 100
    
    # Calculamos el acumulado sobre la lista ya ordenada
    final_report['cum_pct'] = final_report['pct'].cumsum()
    
    # --- 4. VISUALIZACIÓN ---
    # Ajustamos opciones para ver todas las filas en consola
    pd.set_option('display.max_rows', None)
    
    print(f"{'Feature':<25} | {'Gain':<12} | {'% Part.':<8} | {'% Acum.':<8}")
    print("-" * 65)
    
    for _, row in final_report.iterrows():
        print(f"{row['feature']:<25} | {row['gain']:<12.2f} | {row['pct']:<8.2f} | {row['cum_pct']:<8.2f}")
    
    # --- 5. EXPORTAR A CSV (OPCIONAL) ---
    final_report.to_csv("feature_importance_report.csv", index=False)
    print("\n✅ Reporte completo generado y guardado como 'feature_importance_report.csv'")

## 3.2 CatBoost

In [ ]:
%%time
flag_cat = 0
if flag_cat == 1:
    from catboost import CatBoostRegressor, Pool

    import shutil
    from pathlib import Path
    
    # Definir la ruta base de los modelos
    models_dir = Path("/kaggle/working/models")
    
    if models_dir.exists():
        # Iterar sobre las carpetas que empiezan con 'cat'
        cat_models = list(models_dir.glob("cat*"))
        
        if cat_models:
            for folder in cat_models:
                shutil.rmtree(folder)
                print(f"✅ Directorio '{folder.name}' eliminado exitosamente.")
        else:
            print("ℹ️ No se encontraron modelos que empiecen con 'cat' para borrar.")
    else:
        print(f"ℹ️ El directorio '{models_dir}' no existe, se creará al entrenar.")

    # Ejemplo de parámetros para forzar un mejor aprendizaje
    cb_params_base = dict(
        iterations=8000, 
        depth=6,              # Bajamos de 7 a 6 para reducir complejidad
        l2_leaf_reg=5.0,      # Aumentamos regularización para evitar ruido
        min_data_in_leaf=20,  # Un poco más conservador
        border_count=128,     # Reducimos de 254 a 128 (estándar más estable)
        loss_function="RMSE", 
        task_type=DEVICE, 
        devices=CATBOOST_DEVICES,
        od_type="Iter", 
        od_wait=300, 
        verbose=0,
    )

    cb_params_list = [
        # Modelo 1: Configuración estándar equilibrada
        dict(
            iterations=8000, depth=6, l2_leaf_reg=5.0,
            min_data_in_leaf=20, border_count=128,
            learning_rate=0.05, loss_function="RMSE", 
            task_type=DEVICE, devices=CATBOOST_DEVICES,
            od_type="Iter", od_wait=300, verbose=0
        ),
        # Modelo 2: Más profundo (captura detalles sutiles del yacimiento)
        dict(
            iterations=8000, depth=7, l2_leaf_reg=7.0,
            min_data_in_leaf=25, border_count=128,
            learning_rate=0.03, loss_function="RMSE", 
            task_type=DEVICE, devices=CATBOOST_DEVICES,
            od_type="Iter", od_wait=300, verbose=0
        ),
        # Modelo 3: Menos profundo (más rápido, mayor regularización, evita overfitting)
        dict(
            iterations=8000, depth=5, l2_leaf_reg=3.0,
            min_data_in_leaf=15, border_count=128,
            learning_rate=0.08, loss_function="RMSE", 
            task_type=DEVICE, devices=CATBOOST_DEVICES,
            od_type="Iter", od_wait=300, verbose=0
        )
    ]
    
    # 1. Definición de la función (Bloque indentado a 4 espacios)
    def train_catboost(params_input, name):
        # 1. Copia y limpieza FORZADA de parámetros
        params = params_input.copy()
        
        # Corrección crítica para evitar el error de Key 'cpu'
        if 'task_type' in params:
            params['task_type'] = params['task_type'].upper()
        y_log = apply_log(y)
        path = Path("/kaggle/working/models") / name
        
        oof_preds_log = np.zeros(len(train_df), dtype=np.float32)
        val_mask = np.zeros(len(train_df), dtype=bool)
        test_preds = np.zeros(len(X_test), dtype=np.float32)
        models = []
        fold_scores = []
        
        #N_SPLITS = 2
        tscv = TimeSeriesSplit(n_splits=N_SPLITS)
        
        print(f"Training {name} (CatBoost)")
        
        for fold_idx, (train_idx, valid_idx) in enumerate(tscv.split(X)):
            train_pool = Pool(X.iloc[train_idx], label=y_log.iloc[train_idx])
            valid_pool = Pool(X.iloc[valid_idx], label=y_log.iloc[valid_idx])
            
            m = CatBoostRegressor(**params)
            m.fit(train_pool, eval_set=valid_pool, verbose=False)
            
            models.append(m)
            preds_fold_log = m.predict(X.iloc[valid_idx]).astype(np.float32)
            oof_preds_log[valid_idx] = preds_fold_log
            val_mask[valid_idx] = True
            
            test_preds += revert_log_clipped(m.predict(X_test).astype(np.float32)) / N_SPLITS
            
            score = root_mean_squared_error(y.iloc[valid_idx], revert_log_clipped(preds_fold_log))
            fold_scores.append(score)
            print(f"--- Fold {fold_idx} RMSE: {score:.4f}")
        
        valid_indices = np.where(val_mask)[0]
        overall_score = root_mean_squared_error(y.iloc[valid_indices], revert_log_clipped(oof_preds_log[valid_indices]))
        print(f"\nOverall RMSE (real): {overall_score:.4f}")
        
        os.makedirs(path, exist_ok=True)
        joblib.dump(models, path / "models.pkl")
        joblib.dump(oof_preds_log, path / "oof_preds.pkl")
        
        return oof_preds_log, test_preds, overall_score, fold_scores
    
    # 2. Bucle de ejecución (Nivel de indentación igual al de 'def train_catboost')
    for i in range(3):
        print(f"--- Iniciando entrenamiento de catboost-{i + 1} ---")
        
        _oof_preds, _test_preds, _overall_score, _fold_scores = train_catboost(
        cb_params_list[i],  # <--- Ahora esto funcionará perfecto
        f"catboost-{i + 1}"
        )
        oof_preds[f"catboost-{i + 1}"] = _oof_preds
        test_preds[f"catboost-{i + 1}"] = _test_preds
        overall_scores[f"catboost-{i + 1}"] = _overall_score
        fold_scores[f"catboost-{i + 1}"] = _fold_scores
        
        print(f"--- Finalizado catboost-{i + 1} | RMSE: {_overall_score:.4f} ---\n\n")

## 3.3 XGBoost (NEW v33 - replaces TabICL for deterministic diversity)


# 4. Hill climbing

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    oof_preds = pd.DataFrame(oof_preds)
    test_preds = pd.DataFrame(test_preds)

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Aseguramos que tenemos DataFrames de los modelos
    oof_preds_df = pd.DataFrame(oof_preds)
    test_preds_df = pd.DataFrame(test_preds)
    
    # 2. DIAGNÓSTICO DE ESCALA (Crucial)
    # Imprime esto para ver si LGBM está en la misma escala que los demás
    _dbg(f"Estadísticas de predicciones (OOF):\n{oof_preds_df.describe()}")
    
    # 3. NORMALIZACIÓN/REVERSIÓN UNIFICADA
    # Asegúrate de aplicar revert_log a TODOS los modelos ANTES del DataFrame
    oof_preds_real = pd.DataFrame(index=oof_preds_df.index)
    test_preds_real = pd.DataFrame(index=test_preds_df.index)
    
    for col in oof_preds_df.columns:
        # Si tus modelos entrenaron en log, revert_log debe devolver el valor real
        # Verifica que revert_log NO tenga un comportamiento distinto para LGBM
        val_real = revert_log_clipped(oof_preds_df[col].values)
        oof_preds_real[col] = val_real
        test_preds_real[col] = revert_log_clipped(test_preds_df[col].values)
    
    # 4. PASO EXTRA: Limpieza de outliers en la escala real
    # Si un modelo lanzó un valor absurdo por error, esto evitará que el score se dispare
    oof_preds_real = oof_preds_real.clip(y.min(), y.max() * 1.5)

In [ ]:
%%time
flag_xgb = 0
if flag_xgb == 1:
    import xgboost as xgb
    import gc
    import shutil
    
    # 1. Limpieza selectiva de modelos XGB
    models_dir = Path("/kaggle/working/models")
    if models_dir.exists():
        xgb_models = list(models_dir.glob("xgboost*"))
        for folder in xgb_models:
            shutil.rmtree(folder)
            print(f"✅ Limpiado: {folder.name}")

    def train_xgboost(params_input, name):
        # 1. Preparación
        params = params_input.copy()
        n_est = params.pop('n_estimators', 5000)
        y_log = apply_log(y)
        path = Path("/kaggle/working/models") / name
        
        oof_log = np.zeros(len(train_df), np.float32)
        val_mask = np.zeros(len(train_df), dtype=bool)
        tst_log = np.zeros(len(X_test), np.float32)
        models = []; fold_sc = []

        #N_SPLITS = 2
        tscv = TimeSeriesSplit(n_splits=N_SPLITS)
        
        _dbg(f"XGB[{name}] training (Strict Range)\n")
        
        for fi, (tr_idx, va_idx) in enumerate(tscv.split(X)):
            _t = time.time()
            d_tr = xgb.DMatrix(X.iloc[tr_idx].values, label=y_log.iloc[tr_idx].values)
            d_va = xgb.DMatrix(X.iloc[va_idx].values, label=y_log.iloc[va_idx].values)
            
            m = xgb.train(
                params, d_tr, num_boost_round=n_est,
                evals=[(d_va, "va")], early_stopping_rounds=250,
                verbose_eval=False
            )
            models.append(m)
            
            # Predicción con CLIP para forzar el rango de tus datos logarítmicos
            raw_preds = m.predict(d_va, iteration_range=(0, m.best_iteration + 1))
            preds = np.clip(raw_preds, 0, 6) # Ajusta según tu rango logarítmico real
            
            oof_log[va_idx] = preds
            val_mask[va_idx] = True
            
            # Predicción en test con clip
            tst_preds = m.predict(xgb.DMatrix(X_test.values), iteration_range=(0, m.best_iteration + 1))
            tst_log += np.clip(tst_preds, 0, 6) / N_SPLITS
            
            sc = root_mean_squared_error(y.iloc[va_idx], revert_log_clipped(preds))
            fold_sc.append(sc)
            _dbg(f"  XGB[{name}] fold {fi}: RMSE(real)={sc:.4f} | took={time.time()-_t:.1f}s")
        
        # 2. Evaluación final
        valid_indices = np.where(val_mask)[0]
        ov_real = root_mean_squared_error(y.iloc[valid_indices], revert_log_clipped(oof_log[valid_indices]))
        _dbg(f"\nOverall RMSE Final (real): {ov_real:.4f}")
        
        os.makedirs(path, exist_ok=True)
        joblib.dump(models, path / "models.pkl")
        joblib.dump(oof_log, path / "oof_preds.pkl")
        
        return revert_log_clipped(oof_log), revert_log_clipped(tst_log), ov_real, fold_sc

    # Parámetros robustos para ensamble
    xgb_params = [
        dict(objective='reg:squarederror', n_estimators=5000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method='hist'),
        dict(objective='reg:squarederror', n_estimators=5000, learning_rate=0.03, max_depth=7, subsample=0.7, colsample_bytree=0.7, tree_method='hist'),
        dict(objective='reg:squarederror', n_estimators=5000, learning_rate=0.07, max_depth=5, subsample=0.9, colsample_bytree=0.9, tree_method='hist')
    ]

    # 3. Ejecución
    if RUN_XGB:
        for i in range(3):
            _name = f"xgboost-{i+1}"
            _oof, _tst, _ov, _fs = train_xgboost(xgb_params[i], _name)
            oof_preds[_name] = _oof
            test_preds[_name] = _tst
            overall_scores[_name] = _ov
            fold_scores[_name] = _fs
    else:
        _dbg("RUN_XGB=False; skipping XGB")

In [ ]:
print(f"Valor mínimo de Y: {y.min()}")
print(f"Valor máximo de Y: {y.max()}")
print(f"Cantidad de valores negativos en Y: {(y < 0).sum()}")
print(f"El SHIFT_VAL actual es: {SHIFT_VAL}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error

# Definición de la función de Hill Climbing
def proper_hill_climb(oof_df, test_df_blend, y_arr, precision=0.001, score_dp=3, max_iters=3000):
    cols = list(oof_df.columns)
    oof_arr = {c: oof_df[c].values.astype(np.float64) for c in cols}
    tst_arr = {c: test_df_blend[c].values.astype(np.float64) for c in cols}
    sc_single = {c: root_mean_squared_error(y_arr, oof_arr[c]) for c in cols}
    best_m = min(sc_single, key=sc_single.get)
    w = {c: 0.0 for c in cols}
    w[best_m] = 1.0
    blend = oof_arr[best_m].copy()
    total = 1.0
    best_score = sc_single[best_m]
    
    step_schedule = [s for s in (0.5, 0.25, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001) if s >= precision]
    iter_n = 0
    for step in step_schedule:
        while iter_n < max_iters:
            new_total = total + step
            best_pick = None
            best_trial_score = best_score
            best_trial_blend = None
            for m in cols:
                trial_blend = (blend * total + step * oof_arr[m]) / new_total
                trial_score = root_mean_squared_error(y_arr, trial_blend)
                if (round(trial_score, score_dp) < round(best_trial_score, score_dp) and trial_score < best_trial_score):
                    best_trial_score = trial_score
                    best_pick = m
                    best_trial_blend = trial_blend
            if best_pick is None: break
            w[best_pick] += step
            blend = best_trial_blend
            total = new_total
            best_score = best_trial_score
            iter_n += 1
        if iter_n >= max_iters: break
    
    norm_w = {k: v / total for k, v in w.items()}
    hc_oof = blend.astype(np.float32)
    hc_tst = np.zeros(len(test_df_blend), np.float64)
    for k, v in norm_w.items():
        if v > 0: hc_tst += v * tst_arr[k]
    return hc_oof, hc_tst.astype(np.float32), float(best_score), norm_w

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    from sklearn.metrics import root_mean_squared_error, mean_squared_error
    from sklearn.linear_model import Ridge
    import pandas as pd
    import numpy as np
    
    # 1. Función para limpiar datos de valores NaN o Infinitos
    def sanitize_df(df):
        return df.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # 2. Preparación de datos (con sanitización)
    oof_preds_df = pd.DataFrame(oof_preds).pipe(sanitize_df)
    test_preds_df = pd.DataFrame(test_preds).pipe(sanitize_df)
    y_clean = pd.Series(y).fillna(0).values
    
    oof_preds_real = pd.DataFrame(index=oof_preds_df.index)
    test_preds_real = pd.DataFrame(index=test_preds_df.index)
    
    # Transformación y ajuste inicial de escala
    for col in oof_preds_df.columns:
        oof_preds_real[col] = np.expm1(oof_preds_df[col].clip(-20, 20)) - SHIFT_VAL
        test_preds_real[col] = np.expm1(test_preds_df[col].clip(-20, 20)) - SHIFT_VAL
        
        # Ajuste de Bias individual por modelo para alinear escalas base
        bias = oof_preds_real[col].mean() - y_clean.mean()
        oof_preds_real[col] -= bias
        test_preds_real[col] -= bias
    
    # Sanitizamos los dataframes
    oof_preds_real = sanitize_df(oof_preds_real)
    test_preds_real = sanitize_df(test_preds_real)
    
    # 3. Ejecución del Hill Climbing
    hc_oof_preds, hc_test_preds, hc_score, hc_weights = proper_hill_climb(
        oof_preds_real, test_preds_real, y_clean
    )
    
    # 4. Calibración de Bias Global
    # Calculamos el error residual del ensamble frente al target real
    global_bias = hc_oof_preds.mean() - y_clean.mean()
    
    # Aplicamos la corrección de bias antes del Meta-Ajuste
    hc_oof_calibrated = hc_oof_preds - global_bias
    hc_test_calibrated = hc_test_preds - global_bias
    
    # 5. Meta-Ajuste no lineal (Ridge) sobre datos centrados
    X_meta = pd.DataFrame({'blend': hc_oof_calibrated})
    test_meta_input = pd.DataFrame({'blend': hc_test_calibrated})
    
    # 5. Meta-Ajuste (Ridge) SIN INTERCEPTO
    # Al poner fit_intercept=False, Ridge no puede añadir ese "sesgo" de 11.78
    meta_model = Ridge(alpha=1.0, fit_intercept=False) 
    
    # Ajustamos para que el modelo solo aprenda el factor de escala
    meta_model.fit(X_meta, y_clean)
    
    # 6. Predicción y Corrección Forzada
    oof_final = meta_model.predict(X_meta)
    test_final = meta_model.predict(test_meta_input)
    
    # CALIBRACIÓN POST-PREDICCIÓN (Forzamos la media a ser la del target)
    oof_final = oof_final - (oof_final.mean() - y_clean.mean())
    test_final = test_final - (test_final.mean() - y_clean.mean())
    
    # Clipping final
    final_oof = np.clip(oof_final, y_clean.min(), y_clean.max())
    final_test = np.clip(test_final, y_clean.min(), y_clean.max())
    
    # 7. Cálculo y Reporte
    rmse_final = np.sqrt(mean_squared_error(y_clean, final_oof))
    overall_scores["hill-climbing"] = rmse_final
    test_preds["hill-climbing"] = final_test
    
    print(f"--- RESULTADOS FINALES CON CALIBRACIÓN DE BIAS ---")
    print(f"Bias global corregido: {global_bias:.4f}")
    print(f"RMSE Final con Meta-Ajuste: {rmse_final:.4f}")
    print(f"Media final predicha: {final_test.mean():.4f} (Target real: {y_clean.mean():.4f})")

In [ ]:
def apply_pp(df, md, pd_, alpha, tau, w_pf):
    d = md * (1 - w_pf) + pd_ * w_pf
    if tau:
        d *= (1. - np.exp(-np.maximum(df['md_since'].values, 0.) / tau))
    return d * alpha
 
 
def sg_smooth(df, col, sg_w=17, sg_p=3):
    df = df.copy()
    for _, g in df.groupby('well', sort=False):
        v = g[col].values
        n = len(v)
        wl = min(sg_w, n)
        if wl % 2 == 0: wl -= 1
        if wl >= sg_p + 2: v = savgol_filter(v, wl, sg_p)
        df.loc[g.index, col] = v
    return df

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE OPTIMIZACIÓN: INTEGRACIÓN SEGURA ---
    _dbg("Iniciando optimización Optuna (Rango controlado)...")
    
    # 3. Preparación para el Post-procesamiento (escala real)
    base = train_df['last_known_tvt'].values
    ytrue = y.values + base
    pf_oof = (train_df['pf_ancc'].values - base)
    
    
    def objective(trial):
        # Rango de alpha cercano a 1.0 para no escalar agresivamente
        alpha = trial.suggest_float('alpha', 0.90, 1.10, step=0.01)
        
        # Tau controlado para una curva de decaimiento suave
        tau = trial.suggest_int('tau', 10, 200, step=10)
        
        # w_pf (Física) muy restringido para no sesgar la media a -30
        # Empezamos con un rango pequeño para ver si la física ayuda sin dañar
        w_pf = trial.suggest_float('w_pf', 0.0, 0.10, step=0.005)
        
        # Predicción corregida
        d = apply_pp(train_df, hc_oof_preds, pf_oof, alpha, tau, w_pf)
        
        # RMSE contra la verdad absoluta (escala real)
        score = root_mean_squared_error(ytrue, base + d)
        return score
    
    # Configuración del estudio
    sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=50)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=500)
    
    best_params = study.best_params
    _dbg(f"Optimización finalizada. Mejor RMSE: {study.best_value:.4f}")
    _dbg(f"Mejores parámetros: {best_params}")
    
    # Aplicación final en el test (Inferencia)
    pf_test = test_df['pf_ancc'].values - test_df['last_known_tvt'].values
    test_df['pred'] = test_df['last_known_tvt'].values + apply_pp(
        test_df, 
        hc_test_preds, 
        pf_test, 
        best_params['alpha'], 
        best_params['tau'], 
        best_params['w_pf']
    )

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE MAESTRO: OPTIMIZACIÓN Y GENERACIÓN ---
    _dbg("Iniciando optimización Optuna (Rango controlado)...")
    
    # 1. Definición de variables en escala real
    base = train_df['last_known_tvt'].values
    ytrue = y.values + base
    pf_oof = (train_df['pf_ancc'].values - base)
    
    # 2. Definición del objetivo (Optimizando sobre la escala real)
    def objective(trial):
        alpha = trial.suggest_float('alpha', 0.90, 1.10, step=0.01)
        tau = trial.suggest_int('tau', 10, 200, step=10)
        w_pf = trial.suggest_float('w_pf', 0.0, 0.10, step=0.005)
        
        # IMPORTANTE: Si tus modelos base (hc_oof_preds) fueron entrenados en log,
        # descomenta la siguiente línea para convertirlos a escala real antes del PP:
        # d_input = np.expm1(hc_oof_preds) 
        d_input = hc_oof_preds # Si ya es escala real, usa esto
        
        d = apply_pp(train_df, d_input, pf_oof, alpha, tau, w_pf)
        
        return root_mean_squared_error(ytrue, base + d)
    
    # 3. Lanzamiento del estudio
    sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=50)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=500)
    
    best_params = study.best_params
    _dbg(f"Optimización finalizada. Mejor RMSE: {study.best_value:.4f}")
    _dbg(f"Mejores parámetros: {best_params}")
    
    # 4. Inferencia final (Asegurando la misma lógica del objetivo)
    pf_test = test_df['pf_ancc'].values - test_df['last_known_tvt'].values
    
    # Nuevamente, si entrenaste en log, usa np.expm1(hc_test_preds)
    d_test_input = hc_test_preds 
    
    test_df['pred'] = test_df['last_known_tvt'].values + apply_pp(
        test_df, 
        d_test_input, 
        pf_test, 
        best_params['alpha'], 
        best_params['tau'], 
        best_params['w_pf']
    )
    
    # 5. Suavizado final (Post-procesamiento de señales)
    test_df = sg_smooth(test_df, 'pred')

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE FINAL: RETORNO A LA ESTADÍSTICA PURA ---
    _dbg("Desactivando corrección física agresiva; usando Ensamble Puro + Suavizado.")
    
    # 1. Volvemos al resultado del Hill Climbing (que sabemos que es RMSE ~7.5)
    # Esto ignora el sesgo negativo de -30 que introducía el Optuna
    test_df['pred'] = test_df['last_known_tvt'].values + hc_test_preds
    
    # 2. Aplicamos suavizado (el 'SG' que ya demostró funcionar)
    test_df = sg_smooth(test_df, 'pred')
    
    # 3. Preparación de la submission
    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub = sample_sub[["id"]].merge(
        test_df[["id", "pred"]].rename(columns={"pred": "tvt"}),
        on="id", how="left"
    )
    
    # Relleno de seguridad
    _fallback = float(train_df["last_known_tvt"].mean() + train_df["target"].mean())
    sub["tvt"] = sub["tvt"].fillna(_fallback)
    
    # 4. Guardado final
    _out = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
    sub[["id", "tvt"]].to_csv(_out, index=False)
    _dbg(f"Submission generada exitosamente con Ensamble Puro. Archivo en: {_out}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE DE VERIFICACIÓN: RMSE ANTES DE EXPORTAR ---
    _dbg("Calculando RMSE final de la predicción en entrenamiento...")
    
    # 1. Calculamos la predicción en el set de entrenamiento
    # Usamos el mismo pipeline que definimos para test
    train_df['pred'] = train_df['last_known_tvt'].values + hc_oof_preds
    train_df = sg_smooth(train_df, 'pred')
    
    # 2. RMSE Real (Train vs Target)
    final_rmse = root_mean_squared_error(ytrue, train_df['pred'].values)
    
    # 3. Verificación de sesgo (para asegurarnos de que no sea -30)
    media_pred = train_df['pred'].mean()
    _dbg(f"--------------------------------------------------")
    _dbg(f"RMSE Final calculado: {final_rmse:.4f}")
    _dbg(f"Media de tus predicciones: {media_pred:.4f}")
    _dbg(f"Media del target real: {ytrue.mean():.4f}")
    _dbg(f"--------------------------------------------------")
    
    if abs(media_pred - ytrue.mean()) > 10:
        _dbg("ALERTA: Hay un sesgo importante en la media. Revisa la escala.")
    else:
        _dbg("Consistencia de media: OK.")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE DE NORMALIZACIÓN ANTES DEL HILL CLIMBING ---
    _dbg("Aplicando normalización Z-Score antes del Ensamble...")
    
    def z_score_normalize(df):
        # Restamos la media y dividimos por la desviación estándar
        return (df - df.mean()) / df.std()
    
    # Normalizamos los oof_preds y test_preds individualmente
    oof_preds_norm = oof_preds.apply(z_score_normalize)
    test_preds_norm = test_preds.apply(z_score_normalize)
    
    # Ahora el Hill Climbing trabajará con datos normalizados (media 0, std 1)
    hc_oof_norm, hc_test_norm, hc_score, hc_weights = proper_hill_climb(
        oof_preds_norm, test_preds_norm, z_score_normalize(y).values
    )
    
    # RECONSTRUCCIÓN FINAL: Volvemos a la escala original (11469)
    # Multiplicamos por la std del target y sumamos la media del target
    target_mean = y.mean()
    target_std = y.std()
    
    hc_oof_preds = (hc_oof_norm * target_std) + target_mean
    hc_test_preds = (hc_test_norm * target_std) + target_mean
    
    _dbg("Normalización completada. RMSE debería estar ahora en su lugar.")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE FINAL: PROMEDIO PONDERADO ROBUSTO (SIN HILL CLIMBING) ---
    _dbg("Iniciando promedio simple de los 3 modelos base...")
    
    # 1. Aseguramos que oof_preds y test_preds sean DataFrames limpios
    oof_df = pd.DataFrame(oof_preds)
    test_df_preds = pd.DataFrame(test_preds)
    
    # 2. Definimos pesos iguales (o puedes ajustarlos según tu criterio)
    # Como tus modelos base dieron RMSE de 7.8, 7.5 y 7.7, son muy similares.
    # Pesos sugeridos para promediar:
    weights = [0.33, 0.34, 0.33] 
    
    # 3. Calculamos el promedio ponderado
    # Esto evita que un solo valor extremo (NaN o Inf) destruya la predicción
    final_oof = oof_df.mul(weights).sum(axis=1)
    final_test = test_df_preds.mul(weights).sum(axis=1)
    
    # 4. Inferencia final (Escala Real)
    # Sumamos la base para volver a la escala 11469
    test_df['pred'] = test_df['last_known_tvt'].values + final_test
    
    # 5. Suavizado final (esto no toca la media, solo elimina ruido)
    test_df = sg_smooth(test_df, 'pred')
    
    # 6. Guardado de Submission
    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub = sample_sub[["id"]].merge(
        test_df[["id", "pred"]].rename(columns={"pred": "tvt"}),
        on="id", how="left"
    )
    
    # Relleno de seguridad contra NaNs
    sub["tvt"] = sub["tvt"].fillna(float(train_df["last_known_tvt"].mean() + train_df["target"].mean()))
    
    _out = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
    sub[["id", "tvt"]].to_csv(_out, index=False)
    
    _dbg(f"Submission generada. Media final: {test_df['pred'].mean():.4f}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE DE PROMEDIO AUTOMÁTICO (A PRUEBA DE ERRORES) ---
    _dbg("Iniciando promedio simple automático...")
    
    # 1. Aseguramos que trabajamos con DataFrames limpios
    oof_df = pd.DataFrame(oof_preds)
    test_df_preds = pd.DataFrame(test_preds)
    
    # 2. Promedio simple de TODAS las columnas (axis=1)
    # Esto ignora cuántas columnas tengas, las suma y divide por el total.
    # .mean(axis=1) es más seguro que .mul() cuando no sabes el número de columnas.
    final_oof = oof_df.mean(axis=1)
    final_test = test_df_preds.mean(axis=1)
    
    # 3. Inferencia final (Escala Real)
    # Usamos .values para asegurar alineación exacta de índices
    test_df['pred'] = test_df['last_known_tvt'].values + final_test.values
    
    # 4. Suavizado final (SG)
    test_df = sg_smooth(test_df, 'pred')
    
    # 5. Generación de Submission
    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub = sample_sub[["id"]].merge(
        test_df[["id", "pred"]].rename(columns={"pred": "tvt"}),
        on="id", how="left"
    )
    
    # Relleno de seguridad contra NaNs
    fill_val = float(train_df["last_known_tvt"].mean() + train_df["target"].mean())
    sub["tvt"] = sub["tvt"].fillna(fill_val)
    
    # 6. Guardado
    _out = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
    sub[["id", "tvt"]].to_csv(_out, index=False)
    
    _dbg(f"Submission generada. Columnas procesadas: {test_df_preds.shape[1]}")
    _dbg(f"Media final de la predicción: {test_df['pred'].mean():.4f}")
    
    
    # --- BLOQUE DE VALIDACIÓN Y RMSE FINAL ---
    _dbg("Calculando RMSE final sobre entrenamiento...")
    
    # 1. Obtenemos el ensamble sobre el set de entrenamiento
    # (Usamos el promedio simple que ya funcionó)
    oof_df = pd.DataFrame(oof_preds)
    train_pred_raw = oof_df.mean(axis=1).values
    train_df['pred'] = train_df['last_known_tvt'].values + train_pred_raw
    
    # 2. Suavizado para coherencia temporal
    train_df = sg_smooth(train_df, 'pred')
    
    # 3. Cálculo de RMSE contra el Target real
    # Usamos ytrue = y.values + base (definido anteriormente)
    final_rmse = root_mean_squared_error(ytrue, train_df['pred'].values)
    
    _dbg(f"--------------------------------------------------")
    _dbg(f"RMSE Final (Local): {final_rmse:.4f}")
    _dbg(f"Media Predicción:   {train_df['pred'].mean():.4f}")
    _dbg(f"Media Target Real:  {ytrue.mean():.4f}")
    _dbg(f"Diferencia (Bias):  {train_df['pred'].mean() - ytrue.mean():.4f}")
    _dbg(f"--------------------------------------------------")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE DE LIMPIEZA Y CÁLCULO DE RMSE REAL ---
    _dbg("Limpiando NaNs e Infinitos para validación...")
    
    # 1. Convertimos a DataFrame y reemplazamos NaNs/Infs
    oof_df = pd.DataFrame(oof_preds)
    oof_df = oof_df.replace([np.inf, -np.inf], np.nan).fillna(oof_df.mean())
    
    # 2. Calculamos la predicción limpia
    train_pred_raw = oof_df.mean(axis=1).values
    train_df['pred'] = train_df['last_known_tvt'].values + train_pred_raw
    train_df = sg_smooth(train_df, 'pred')
    
    # 3. Cálculo de RMSE sin NaNs
    # Filtramos ytrue y pred para que solo comparen valores válidos
    mask = ~np.isnan(train_df['pred'].values) & ~np.isnan(ytrue)
    final_rmse = root_mean_squared_error(ytrue[mask], train_df['pred'].values[mask])
    
    _dbg(f"--------------------------------------------------")
    _dbg(f"RMSE Final (Limpio): {final_rmse:.4f}")
    _dbg(f"Media Predicción:    {train_df['pred'].values[mask].mean():.4f}")
    _dbg(f"Media Target Real:   {ytrue[mask].mean():.4f}")
    _dbg(f"--------------------------------------------------")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE: EXTRACCIÓN AUTOMÁTICA DE RMSE ---
    _dbg("Extrayendo RMSE directamente de los logs...")
    
    # Asumimos que tienes una lista de objetos o diccionarios donde guardaste los resultados de cada fold
    # o que los modelos fueron entrenados y evaluados en un loop.
    # Aquí calculamos el RMSE de cada modelo base dinámicamente:
    rmse_por_modelo = []
    
    for i in range(oof_preds.shape[1]):
        # Calculamos el error directamente sobre los OOF (Out-of-Fold) de cada modelo
        col_name = oof_preds.columns[i]
        val = root_mean_squared_error(y.values, oof_preds[col_name].values)
        rmse_por_modelo.append(val)
    
    # Cálculo dinámico de pesos
    inv_rmse = np.array([1.0 / e for e in rmse_por_modelo])
    pesos = inv_rmse / inv_rmse.sum()
    
    _dbg(f"Pesos calculados automáticamente: {pesos.tolist()}")
    _dbg(f"RMSEs detectados: {rmse_por_modelo}")
    
    # Aplicación del ensamble
    final_test_vals = np.average(test_df_preds.values, axis=1, weights=pesos)
    test_df['pred'] = test_df['last_known_tvt'].values + final_test_vals
    
    # Corrección de bias (Centrado matemático)
    bias_final = (test_df['last_known_tvt'].values + final_test_vals).mean() - ytrue.mean()
    test_df['pred'] = test_df['pred'] - bias_final
    
    # Suavizado y guardado
    test_df = sg_smooth(test_df, 'pred')

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE DE LIMPIEZA TOTAL Y ENSAMBLE DINÁMICO ---
    _dbg("Limpiando NaNs e Infinitos y calculando pesos...")
    
    # 1. Limpieza de datos (Reemplazo global de NaNs por la media de cada modelo)
    # Esto es vital: si un modelo falla en un pozo, usamos su propio promedio histórico
    oof_clean = oof_preds.replace([np.inf, -np.inf], np.nan).fillna(oof_preds.mean())
    test_clean = test_df_preds.replace([np.inf, -np.inf], np.nan).fillna(test_df_preds.mean())
    
    # 2. Extracción automática de RMSEs (Validación sobre OOF limpio)
    rmse_por_modelo = []
    for col in oof_clean.columns:
        error = root_mean_squared_error(y.values, oof_clean[col].values)
        rmse_por_modelo.append(error)
    
    # 3. Cálculo dinámico de pesos (basado en el RMSE real calculado)
    inv_rmse = np.array([1.0 / e for e in rmse_por_modelo])
    pesos = inv_rmse / inv_rmse.sum()
    
    _dbg(f"RMSEs detectados: {rmse_por_modelo}")
    _dbg(f"Pesos calculados: {pesos.tolist()}")
    
    # 4. Ensamble ponderado sobre los datos limpios
    final_test_vals = np.average(test_clean.values, axis=1, weights=pesos)
    
    # 5. Reconstrucción y Corrección de Sesgo (Bias)
    # Aplicamos el ensamble a la base y centramos la media para eliminar el error sistemático
    pred_raw = test_df['last_known_tvt'].values + final_test_vals
    bias_correction = pred_raw.mean() - ytrue.mean()
    test_df['pred'] = pred_raw - bias_correction
    
    # 6. Suavizado final
    test_df = sg_smooth(test_df, 'pred')
    
    # 7. Verificación final de integridad
    if test_df['pred'].isnull().any():
        _dbg("ALERTA: Aún existen NaNs tras el procesamiento. Sustituyendo por promedio global.")
        test_df['pred'] = test_df['pred'].fillna(test_df['pred'].mean())
    
    _dbg("Proceso de ensamble finalizado con éxito.")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE: ALINEACIÓN ESTRICTA Y ENSAMBLE ---
    _dbg("Alineando dimensiones de modelos y pesos...")
    
    # 1. Obtenemos solo las columnas que usamos para calcular el RMSE
    cols_usadas = oof_clean.columns.tolist()
    
    # 2. Filtramos test_clean para que tenga exactamente las mismas columnas
    test_clean_aligned = test_clean[cols_usadas]
    
    # 3. Verificación de seguridad
    if test_clean_aligned.shape[1] != len(pesos):
        _dbg(f"ERROR: Dimensión mismatch. Pesos: {len(pesos)}, Columnas: {test_clean_aligned.shape[1]}")
    else:
        # 4. Ensamble ponderado con columnas alineadas
        final_test_vals = np.average(test_clean_aligned.values, axis=1, weights=pesos)
        
        # 5. Aplicación
        pred_raw = test_df['last_known_tvt'].values + final_test_vals
        
        # Ajuste final de Bias
        bias_correction = pred_raw.mean() - ytrue.mean()
        test_df['pred'] = pred_raw - bias_correction
        
        # Suavizado y guardado
        test_df = sg_smooth(test_df, 'pred')
        
        # Generación de archivo
        sub = sample_sub[["id"]].merge(
            test_df[["id", "pred"]].rename(columns={"pred": "tvt"}),
            on="id", how="left"
        )
        sub["tvt"] = sub["tvt"].fillna(float(test_df['pred'].mean()))
        
        _out = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
        sub[["id", "tvt"]].to_csv(_out, index=False)
        
        _dbg(f"ÉXITO: Submission generada usando {len(cols_usadas)} modelos.")
        _dbg(f"Media final: {test_df['pred'].mean():.4f}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE FINAL: CÁLCULO DE RMSE DE VALIDACIÓN ---
    _dbg("Calculando RMSE final de validación...")
    
    # 1. Obtenemos el ensamble (ya alineado a 3 modelos)
    # Usamos las mismas columnas usadas para el cálculo de pesos
    final_oof_vals = np.average(oof_clean[cols_usadas].values, axis=1, weights=pesos)
    
    # 2. Aplicamos la misma corrección de bias que hicimos en el test
    pred_oof = train_df['last_known_tvt'].values + final_oof_vals
    bias_correction = pred_oof.mean() - ytrue.mean()
    train_df['pred_final'] = pred_oof - bias_correction
    
    # 3. Suavizado final
    train_df = sg_smooth(train_df, 'pred_final')
    
    # 4. Cálculo final contra ytrue
    # Usamos la máscara de NaN por seguridad
    mask = ~np.isnan(train_df['pred_final'].values) & ~np.isnan(ytrue)
    final_rmse = root_mean_squared_error(ytrue[mask], train_df['pred_final'].values[mask])
    
    _dbg(f"--------------------------------------------------")
    _dbg(f"RMSE Final Obtenido: {final_rmse:.4f}")
    _dbg(f"Modelos utilizados: {len(cols_usadas)}")
    _dbg(f"Media final vs Target: {train_df['pred_final'].values[mask].mean():.4f} vs {ytrue[mask].mean():.4f}")
    _dbg(f"--------------------------------------------------")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 5. En lugar de Ridge, usamos el resultado del Hill Climbing directamente
    # Si el Hill Climbing ya hizo su trabajo, el oof_final es hc_oof_preds
    oof_final = hc_oof_preds
    test_final = hc_test_preds
    
    # 6. Corrección de media "Cero-Sesgo" (Forzada)
    # Esto garantiza al 100% que la media sea 1.2777
    oof_final = oof_final - (oof_final.mean() - y_clean.mean())
    test_final = test_final - (test_final.mean() - y_clean.mean())
    
    # 7. Clipping final estricto
    final_oof = np.clip(oof_final, y_clean.min(), y_clean.max())
    final_test = np.clip(test_final, y_clean.min(), y_clean.max())
    
    # 8. Cálculo de RMSE Final
    rmse_final = np.sqrt(mean_squared_error(y_clean, final_oof))

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Normalización previa al Hill Climbing
    def normalize_preds(df_oof, df_test, y_arr):
        df_oof_norm = df_oof.copy()
        df_test_norm = df_test.copy()
        
        for col in df_oof.columns:
            m, s = df_oof[col].mean(), df_oof[col].std()
            df_oof_norm[col] = (df_oof[col] - m) / s
            df_test_norm[col] = (df_test[col] - m) / s
            
        return df_oof_norm, df_test_norm
    
    # 2. Aplicamos normalización
    oof_real_norm, test_real_norm = normalize_preds(oof_preds_real, test_preds_real, y_clean)
    
    # 3. Ejecutamos Hill Climbing en espacio normalizado
    hc_oof_norm, hc_test_norm, hc_score, hc_weights = proper_hill_climb(
        oof_real_norm, test_real_norm, y_clean
    )
    
    # 4. Des-normalizamos el resultado final para volver a la escala del target
    # Calculamos la media y std del ensamble normalizado para revertir
    final_oof = hc_oof_norm * y_clean.std() + y_clean.mean()
    final_test = hc_test_norm * y_clean.std() + y_clean.mean()
    
    # 5. Clipping y RMSE
    final_oof = np.clip(final_oof, y_clean.min(), y_clean.max())
    rmse_final = np.sqrt(mean_squared_error(y_clean, final_oof))
    print(f"RMSE Final con Normalización: {rmse_final:.4f}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. NO normalices. Trabaja con los datos OOF originales (log-space)
    # Asegúrate de que oof_preds_df NO tenga bias restado ni normalización Z-score.
    # El HC debe hacerse sobre la escala en la que LightGBM aprendió.
    hc_oof_log, hc_test_log, hc_score, hc_weights = proper_hill_climb(
        oof_preds_df, test_preds_df, y_log # <--- Usar el target logarítmico original
    )
    
    # 2. Ahora sí, revertimos todo de una sola vez
    final_oof = np.expm1(hc_oof_log) - SHIFT_VAL
    final_test = np.expm1(hc_test_log) - SHIFT_VAL
    
    # 3. Solo ahora, si hay un sesgo de media, lo corregimos
    # Esto es mucho más seguro que normalizar antes
    final_oof = final_oof - (final_oof.mean() - y.mean())
    
    # 4. Clipping final
    final_oof = np.clip(final_oof, y.min(), y.max())
    rmse_final = np.sqrt(mean_squared_error(y, final_oof))

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Asegurar que trabajamos con el mismo índice
    # Asumiendo que 'train_df' es tu DataFrame ya limpio (con dropna y sin outliers)
    # Solo usaremos las filas que sobrevivieron al filtro anterior.
    
    # Sincronizamos 'y_log' con el índice de 'train_df'
    y_log_sincronizado = y_log.loc[train_df.index]
    
    # Sincronizamos las predicciones (oof_preds_df) con el mismo índice
    # Esto es vital: si 'oof_preds' es una lista de arrays, conviértelo a un DF 
    # cuyo índice coincida con el de 'train_df'
    oof_preds_df = pd.DataFrame(oof_preds, index=train_df.index)
    
    # 2. Verificación de seguridad (esto evitará el ValueError)
    if len(oof_preds_df) != len(y_log_sincronizado):
        print(f"⚠️ ERROR: Longitudes diferentes.")
        print(f"oof_preds_df: {len(oof_preds_df)}, y_log: {len(y_log_sincronizado)}")
        # Forzamos la intersección de índices
        comunes = oof_preds_df.index.intersection(y_log_sincronizado.index)
        oof_preds_df = oof_preds_df.loc[comunes]
        y_log_sincronizado = y_log_sincronizado.loc[comunes]
        print(f"✅ Sincronizado a: {len(oof_preds_df)} filas.")
    
    # 3. Ahora sí, ejecutamos el Hill Climbing en espacio logarítmico
    hc_oof_log, hc_test_log, hc_score, hc_weights = proper_hill_climb(
        oof_preds_df, test_preds_df, y_log_sincronizado.values
    )

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Limpieza estricta y sincronización
    # Convertimos todo a numeric y limpiamos NaNs/Infs en un solo paso
    oof_clean = oof_preds_df.apply(pd.to_numeric, errors='coerce').fillna(0).replace([np.inf, -np.inf], 0)
    y_clean = pd.Series(y_log_sincronizado).fillna(0).replace([np.inf, -np.inf], 0).values
    
    # 2. Verificación final de integridad (si esto imprime True, algo sigue mal)
    print(f"¿Hay NaNs en oof_clean?: {oof_clean.isna().any().any()}")
    print(f"¿Hay NaNs en y_clean?: {np.isnan(y_clean).any()}")
    
    # 3. Ejecución del Hill Climbing en espacio logarítmico
    # Usamos oof_clean y y_clean que ya están garantizados sin NaN
    hc_oof_log, hc_test_log, hc_score, hc_weights = proper_hill_climb(
        oof_clean, test_preds_df.fillna(0).replace([np.inf, -np.inf], 0), y_clean
    )
    
    # 4. Reversión logarítmica final
    final_oof = np.expm1(hc_oof_log) - SHIFT_VAL
    final_test = np.expm1(hc_test_log) - SHIFT_VAL
    
    # 5. Ajuste de media final (Post-procesamiento)
    final_oof = final_oof - (final_oof.mean() - y.mean())
    final_test = final_test - (final_test.mean() - y.mean())
    
    # 6. Clipping seguro
    final_oof = np.clip(final_oof, y.min(), y.max())
    final_test = np.clip(final_test, y.min(), y.max())
    
    print(f"RMSE Final obtenido: {np.sqrt(mean_squared_error(y, final_oof)):.4f}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Definir el Meta-Modelo
    # Usamos Ridge porque es robusto ante modelos que "rugen" (predicciones extremas)
    meta_model = Ridge(alpha=10.0, fit_intercept=True) 
    
    # 2. Entrenar el Meta-Modelo usando las predicciones OOF (Out-Of-Fold)
    # OOF son las predicciones que tus modelos hicieron cuando no estaban viendo el target
    meta_model.fit(oof_preds_df, y_clean)
    
    # 3. Predicciones finales
    # El meta-modelo combina los modelos base de forma óptima
    final_oof = meta_model.predict(oof_preds_df)
    final_test = meta_model.predict(test_preds_df)
    
    # 4. Clipping final (obligatorio tras cualquier regresión lineal)
    final_oof = np.clip(final_oof, y_clean.min(), y_clean.max())
    final_test = np.clip(final_test, y_clean.min(), y_clean.max())

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    from sklearn.linear_model import Ridge
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    import numpy as np
    import pandas as pd
    from sklearn.metrics import mean_squared_error
    
    # 1. Limpieza radical: Aseguramos que solo usamos las columnas que tienen datos de modelos base
    # Eliminamos cualquier columna que se llame 'hill-climbing' si existe
    cols_to_drop = ['hill-climbing']
    X_train_df = pd.DataFrame(oof_preds).drop(columns=cols_to_drop, errors='ignore')
    X_test_df = pd.DataFrame(test_preds).drop(columns=cols_to_drop, errors='ignore')
    
    # 2. Asegurar que las columnas sean idénticas y estén en el mismo orden
    # Esto es vital: si el train tiene columnas que el test no, el modelo fallará
    common_cols = X_train_df.columns.intersection(X_test_df.columns)
    X_train_final = X_train_df[common_cols]
    X_test_final = X_test_df[common_cols]
    
    # 3. Definición del Pipeline (ahora con los datos limpios)
    meta_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=10.0))
    ])
    
    # 4. Entrenamiento
    meta_pipeline.fit(X_train_final, y_clean)
    
    # 5. Predicción
    final_oof = meta_pipeline.predict(X_train_final)
    final_test = meta_pipeline.predict(X_test_final)
    
    # 6. Clipping final
    final_oof = np.clip(final_oof, y_clean.min(), y_clean.max())
    final_test = np.clip(final_test, y_clean.min(), y_clean.max())
    
    # 7. Reporte
    rmse_final = np.sqrt(mean_squared_error(y_clean, final_oof))
    print(f"--- META-MODELO RIDGE LIMPIO ---")
    print(f"RMSE Final: {rmse_final:.4f}")
    print(f"Features usados: {len(common_cols)}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    from sklearn.ensemble import HistGradientBoostingRegressor
    
    # 1. Definimos un meta-modelo capaz de capturar la no-linealidad
    # Al ser un árbol, no promediará los resultados, aprenderá patrones
    meta_model_nolineal = HistGradientBoostingRegressor(
        max_iter=100, 
        learning_rate=0.1, 
        max_depth=5, 
        random_state=42
    )
    
    # 2. Entrenamos el meta-modelo sobre los outputs del LGBM (X_train_final)
    meta_model_nolineal.fit(X_train_final, y_clean)
    
    # 3. Predicciones
    final_oof_nl = meta_model_nolineal.predict(X_train_final)
    
    # 4. Verificación de distribución
    print(f"Nueva Desviación Std: {final_oof_nl.std():.4f} (Real: {y_clean.std():.4f})")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Creamos un nuevo dataset que combine:
    #    - Las predicciones de tus 3 modelos (X_train_final)
    #    - Una muestra representativa de tus variables originales (ej. las top 10 importantes)
    #    - IMPORTANTE: Esto debe estar en el mismo DataFrame
    X_stacked = pd.concat([X_train_final, X_train_original_subset], axis=1)
    
    # 2. Usamos un meta-modelo de bosque aleatorio (Random Forest)
    # Los árboles son los mejores para "corregir" el sesgo de otros modelos
    from sklearn.ensemble import RandomForestRegressor
    
    meta_rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=7,
        min_samples_split=10,
        n_jobs=-1,
        random_state=42
    )
    
    # 3. Entrenamiento
    meta_rf.fit(X_stacked, y_clean)
    
    # 4. Predicción
    final_oof_rf = meta_rf.predict(X_stacked)
    
    print(f"Nueva Desviación Std: {final_oof_rf.std():.4f} (Real: {y_clean.std():.4f})")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    def rescale_predictions(preds, target_mean, target_std):
        """
        Fuerza a que las predicciones tengan la media y std del target real.
        Esto recupera la varianza 'perdida' por el meta-modelo.
        """
        # Centrar en 0 y escalar
        preds_norm = (preds - preds.mean()) / preds.std()
        # Re-escalar a los parámetros del target
        return (preds_norm * target_std) + target_mean
    
    # 1. Aplicamos el ajuste sobre tu predicción actual del meta-modelo
    final_oof_rescaled = rescale_predictions(final_oof_nl, y_clean.mean(), y_clean.std())
    
    # 2. Clipping para asegurar que no nos pasamos de los límites reales
    final_oof_rescaled = np.clip(final_oof_rescaled, y_clean.min(), y_clean.max())
    
    # 3. Verificación
    print(f"Nueva Desviación Std: {final_oof_rescaled.std():.4f} (Real: {y_clean.std():.4f})")
    print(f"Nueva Media: {final_oof_rescaled.mean():.4f} (Real: {y_clean.mean():.4f})")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # Comparación final de métricas post-rescalado
    final_metrics = pd.DataFrame({
        'Atributo': ['Media', 'Desviación Std', 'RMSE (Ajustado)'],
        'Target Real': [y_clean.mean(), y_clean.std(), 0.0],
        'Tu Modelo': [final_oof_rescaled.mean(), final_oof_rescaled.std(), np.sqrt(mean_squared_error(y_clean, final_oof_rescaled))]
    })
    
    print("--- REPORTE DE CONSISTENCIA ESTADÍSTICA ---")
    print(final_metrics.to_string(index=False))
    
    # Cálculo final del R2 corregido
    r2_corregido = r2_score(y_clean, final_oof_rescaled)
    print(f"\nCoeficiente de Determinación R²: {r2_corregido:.4f}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    residuos = y_clean - final_oof
    
    plt.figure(figsize=(10, 4))
    # Histograma de residuos: Debe ser una campana de Gauss centrada en 0
    sns.histplot(residuos, kde=True)
    plt.title("Distribución de los Residuos (Debe ser Normal)")
    plt.show()
    
    # Gráfico de dispersión: No debe haber formas (debe parecer una nube aleatoria)
    plt.scatter(final_oof, residuos, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.title("Residuos vs Predicciones (No debe haber patrones)")
    plt.show()

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Ejecutar validación cruzada para demostrar consistencia
    from sklearn.model_selection import cross_val_score
    cv_scores = cross_val_score(meta_pipeline, X_train_final, y_clean, 
                                scoring='neg_root_mean_squared_error', cv=5)
    
    # 2. Extraer pesos (importancia de cada modelo base)
    weights = meta_pipeline.named_steps['ridge'].coef_
    model_names = X_train_final.columns
    
    # 3. Construir la Tabla de Demostración
    results_table = pd.DataFrame({
        'Métrica': ['RMSE Entrenamiento', 'RMSE (Promedio CV)', 'Desviación CV (Std)', 'Bias (Media Residuo)'],
        'Valor': [
            f"{rmse_final:.4f}", 
            f"{-cv_scores.mean():.4f}", 
            f"{cv_scores.std():.4f}", 
            f"{np.mean(y_clean - final_oof):.4f}"
        ]
    })
    
    weights_table = pd.DataFrame({
        'Modelo Base': model_names,
        'Peso Asignado (Importancia)': weights
    })
    
    print("--- TABLA DE VALIDACIÓN ESTADÍSTICA ---")
    print(results_table.to_string(index=False))
    print("\n--- TABLA DE APORTE DE MODELOS (IMPORTANCIA) ---")
    print(weights_table.to_string(index=False))

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Comparación de distribución (esto reemplaza la utilidad del R2)
    results_comparison = pd.DataFrame({
        'Estadístico': ['Media', 'Desviación Std', 'Mínimo', 'Máximo'],
        'Real': [y_clean.mean(), y_clean.std(), y_clean.min(), y_clean.max()],
        'Predicho': [final_oof.mean(), final_oof.std(), final_oof.min(), final_oof.max()]
    })
    
    print("--- COMPARATIVA DE DISTRIBUCIÓN ---")
    print(results_comparison.to_string(index=False))

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error
    
    # 1. Cálculo de métricas adicionales
    mae = mean_absolute_error(y_clean, final_oof)
    medae = median_absolute_error(y_clean, final_oof)
    r2 = r2_score(y_clean, final_oof)
    mape = np.mean(np.abs((y_clean - final_oof) / (y_clean + 1e-9))) * 100 # Error porcentual
    
    # 2. Construcción del reporte tabular
    metrics_report = pd.DataFrame({
        'Métrica': ['RMSE', 'MAE', 'Median AE', 'R² Score', 'MAPE (%)'],
        'Descripción': [
            'Raíz del Error Cuadrático Medio', 
            'Error Absoluto Medio', 
            'Error Absoluto Mediano', 
            'Coeficiente de Determinación', 
            'Error Porcentual Absoluto Medio'
        ],
        'Valor': [f"{rmse_final:.4f}", f"{mae:.4f}", f"{medae:.4f}", f"{r2:.4f}", f"{mape:.2f}%"]
    })
    
    print("--- REPORTE TÉCNICO DE MÉTRICAS DE REGRESIÓN ---")
    print(metrics_report.to_string(index=False))

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    from sklearn.metrics import root_mean_squared_error, mean_squared_error
    from sklearn.linear_model import Ridge
    import pandas as pd
    import numpy as np
    
    # 1. Función para limpiar datos de valores NaN o Infinitos
    def sanitize_df(df):
        # Reemplazamos Infinitos por NaN y luego llenamos todo con 0 (o la mediana)
        return df.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # 2. Preparación de datos (con sanitización obligatoria)
    oof_preds_df = pd.DataFrame(oof_preds).pipe(sanitize_df)
    test_preds_df = pd.DataFrame(test_preds).pipe(sanitize_df)
    y_clean = pd.Series(y).fillna(0).values
    
    oof_preds_real = pd.DataFrame(index=oof_preds_df.index)
    test_preds_real = pd.DataFrame(index=test_preds_df.index)
    
    for col in oof_preds_df.columns:
        # Transformación logarítmica inversa
        oof_preds_real[col] = np.expm1(oof_preds_df[col].clip(-20, 20)) - SHIFT_VAL
        test_preds_real[col] = np.expm1(test_preds_df[col].clip(-20, 20)) - SHIFT_VAL
        
        # Ajuste de Bias
        bias = oof_preds_real[col].mean() - y_clean.mean()
        oof_preds_real[col] -= bias
        test_preds_real[col] -= bias
    
    # Sanitizamos los dataframes finales antes del Hill Climbing
    oof_preds_real = sanitize_df(oof_preds_real)
    test_preds_real = sanitize_df(test_preds_real)
    
    # 3. Ejecución del Hill Climbing
    hc_oof_preds, hc_test_preds, hc_score, hc_weights = proper_hill_climb(
        oof_preds_real, test_preds_real, y_clean
    )
    
    # 4. Meta-Ajuste no lineal (Ridge)
    X_meta = pd.DataFrame({'blend': hc_oof_preds})
    test_meta_input = pd.DataFrame({'blend': hc_test_preds})
    
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(X_meta, y_clean)
    
    # 5. Predicciones finales
    final_oof = np.clip(meta_model.predict(X_meta), y_clean.min(), y_clean.max())
    final_test = np.clip(meta_model.predict(test_meta_input), y_clean.min(), y_clean.max())
    
    # 6. Cálculo y Reporte
    rmse_final = np.sqrt(mean_squared_error(y_clean, final_oof))
    overall_scores["hill-climbing"] = rmse_final
    test_preds["hill-climbing"] = final_test
    
    print(f"--- RESULTADOS FINALES ---")
    print(f"RMSE Final con Meta-Ajuste: {rmse_final:.4f}")
    print(f"Media final predicha: {final_test.mean():.4f} (Target real: {y_clean.mean():.4f})")

In [ ]:
# 1. Diagnóstico de escala
if FLAG_MODEL and not IS_SUBMISSION:
    print(f"Rango del target real: {y_clean.min():.2f} a {y_clean.max():.2f}")
    print(f"Rango del hc_oof_preds (antes de Ridge): {hc_oof_preds.min():.2f} a {hc_oof_preds.max():.2f}")

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    import numpy as np
    import pandas as pd
    from sklearn.metrics import root_mean_squared_error
    import matplotlib.pyplot as plt
    
    # 1. Función de Mapeo de Cuantiles (Alineación de distribución)
    def quantile_mapping(preds, target_vals):
        """
        Fuerza a 'preds' a tener la misma distribución estadística que 'target_vals'.
        Esto elimina errores de escala, sesgos y valores atípicos imposibles.
        """
        # Calculamos el rango percentil de cada predicción (0 a 1)
        preds_rank = pd.Series(preds).rank(pct=True).values
        
        # Obtenemos los valores ordenados del target real
        target_sorted = np.sort(target_vals)
        
        # Mapeamos los percentiles al rango real del target
        mapped_preds = np.interp(preds_rank, np.linspace(0, 1, len(target_sorted)), target_sorted)
        return mapped_preds
    
    # 2. Ejecución del Hill Climbing (obtenemos el blend base)
    # Asegúrate de que hc_oof_preds y hc_test_preds provengan de proper_hill_climb
    hc_oof_preds, hc_test_preds, hc_score, hc_weights = proper_hill_climb(
        pd.DataFrame(oof_preds), pd.DataFrame(test_preds), y.values
    )
    
    # 3. Refinamiento mediante Mapeo de Cuantiles
    # Esto garantiza que final_oof y final_test tengan la misma media y rango que y
    final_oof = quantile_mapping(hc_oof_preds, y.values)
    final_test = quantile_mapping(hc_test_preds, y.values)
    
    # 4. Cálculo de métricas finales
    rmse_final = root_mean_squared_error(y.values, final_oof)
    
    # 5. Registro de resultados
    overall_scores["hill-climbing"] = rmse_final
    test_preds["hill-climbing"] = final_test
    
    # Resultados en consola
    print(f"--- RESULTADOS POR QUANTILE MAPPING ---")
    print(f"RMSE Final: {rmse_final:.4f}")
    print(f"Media final predicha: {final_test.mean():.4f}")
    print(f"Media del Target Real: {y.values.mean():.4f}")
    
    # Visualización para confirmar la alineación de distribuciones
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(y.values, bins=30, alpha=0.5, label='Target Real', color='blue')
    plt.hist(final_oof, bins=30, alpha=0.5, label='Predicción Corregida', color='red')
    plt.legend()
    plt.title('Distribución: Target vs Predicción')
    
    plt.subplot(1, 2, 2)
    plt.scatter(y.values, final_oof, alpha=0.3, s=10)
    plt.plot([y.values.min(), y.values.max()], [y.values.min(), y.values.max()], 'k--')
    plt.title('Correlación Lineal (Ideal)')
    plt.show()

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # Asegúrate de ejecutar esto primero:
    hc_oof_preds, hc_test_preds, hc_score, hc_weights = proper_hill_climb(
        pd.DataFrame(oof_preds), pd.DataFrame(test_preds), y.values
    )
    
    # A. Crear máscara limpia
    # Verificamos si y.values tiene NaNs (posible causa del error previo)
    y_clean = y.values.flatten()
    mask = ~np.isnan(hc_oof_preds) & ~np.isnan(y_clean)
    
    # B. Limpiar
    clean_oof = hc_oof_preds[mask]
    clean_y = y_clean[mask]
    
    # C. Mapeo de cuantiles
    final_oof = np.full_like(hc_oof_preds, np.nan)
    final_oof[mask] = quantile_mapping(clean_oof, clean_y)
    final_test = quantile_mapping(hc_test_preds, clean_y)
    
    # D. Calcular RMSE final
    rmse_final = root_mean_squared_error(clean_y, final_oof[mask])
    
    print(f"--- RESULTADOS POR QUANTILE MAPPING ---")
    print(f"RMSE Final: {rmse_final:.4f}")

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Asegurar importaciones
    from sklearn.linear_model import Ridge
    from sklearn.metrics import root_mean_squared_error
    import pandas as pd
    import numpy as np
    
    # 2. Preparación de datos (Transformación inversa directa)
    # Asumiendo que oof_preds y test_preds son tus DataFrames originales de predicciones
    oof_preds_real = pd.DataFrame(index=oof_preds_df.index)
    test_preds_real = pd.DataFrame(index=test_preds_df.index)
    
    for col in oof_preds_df.columns:
        # Aplicamos la reversión del logaritmo sin sesgo manual
        oof_preds_real[col] = np.expm1(oof_preds_df[col].clip(-20, 20)) - SHIFT_VAL
        test_preds_real[col] = np.expm1(test_preds_df[col].clip(-20, 20)) - SHIFT_VAL
    
    # 3. Meta-Ajuste con Ridge (Aprende la combinación y el Bias)
    # Al pasarle todas las columnas, Ridge encontrará los pesos que minimizan el MSE global
    meta_model = Ridge(alpha=10.0, fit_intercept=True)
    meta_model.fit(oof_preds_real, y)
    
    # 4. Predicciones refinadas
    oof_final = meta_model.predict(oof_preds_real)
    test_final = meta_model.predict(test_preds_real)
    
    # 5. Clipping de seguridad (Usamos 0 como mínimo si el target es positivo)
    # Es más seguro usar 0 si el target real no debería ser negativo
    min_val = max(0, y.min()) 
    final_oof = np.clip(oof_final, min_val, y.max())
    final_test = np.clip(test_final, min_val, y.max())
    
    # 6. Cálculo del RMSE Final
    rmse_final = np.sqrt(mean_squared_error(y, final_oof))
    
    # 7. Registro de resultados
    overall_scores["final_ensemble"] = rmse_final
    test_preds["final_ensemble"] = final_test
    
    print(f"RMSE Final con Meta-Ajuste Ridge: {rmse_final:.4f}")
    print(f"Media final calculada: {final_test.mean():.4f} (Target real: {y.mean():.4f})")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Obtenemos tus predicciones actuales en escala real
    # (Usa la función corregida sin el clip restrictivo)
    preds_real = np.expm1(hc_test_preds) - SHIFT_VAL
    
    # 2. Calculamos el sesgo (Bias)
    bias = preds_real.mean() - y.mean()
    
    # 3. Aplicamos la corrección de sesgo
    preds_calibradas = preds_real - bias
    
    print(f"Media original de predicciones: {preds_real.mean():.4f}")
    print(f"Media de Y real: {y.mean():.4f}")
    print(f"Bias detectado: {bias:.4f}")
    print(f"Nueva media calibrada: {preds_calibradas.mean():.4f}")
    
    # 4. Ahora guarda estas preds_calibradas
    test_preds["hill-climbing"] = preds_calibradas

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1# 1. No hagamos el clip todavía. Veamos qué hay "dentro"
    oof_preds_real = np.expm1(hc_oof_preds) - SHIFT_VAL
    bias = oof_preds_real.mean() - y.mean()
    preds_sin_clip = oof_preds_real - bias
    
    # 2. Vamos a centrar y escalar (Standardization approach)
    # Esto es más robusto que la corrección de sesgo simple
    mean_preds = preds_sin_clip.mean()
    std_preds = preds_sin_clip.std()
    mean_y = y.mean()
    std_y = y.std()
    
    # Escalamos las predicciones para que tengan la misma media y desviación estándar que Y
    preds_rescaladas = (preds_sin_clip - mean_preds) / std_preds * std_y + mean_y
    
    # 3. Ahora calculamos el RMSE con las predicciones rescaladas
    rmse_rescalado = np.sqrt(mean_squared_error(y, preds_rescaladas))
    
    print(f"RMSE tras rescalado: {rmse_rescalado:.4f}")
    print(f"Media tras rescalado: {preds_rescaladas.mean():.4f}")

# 5. Postprocessing

In [ ]:
def apply_pp(df, md, pd_, alpha, tau, w_pf):
    d = md * (1 - w_pf) + pd_ * w_pf
    if tau:
        d *= (1. - np.exp(-np.maximum(df['md_since'].values, 0.) / tau))
    return d * alpha
 
 
def sg_smooth(df, col, sg_w=17, sg_p=3):
    df = df.copy()
    for _, g in df.groupby('well', sort=False):
        v = g[col].values
        n = len(v)
        wl = min(sg_w, n)
        if wl % 2 == 0: wl -= 1
        if wl >= sg_p + 2: v = savgol_filter(v, wl, sg_p)
        df.loc[g.index, col] = v
    return df

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE INTEGRADO: HILL CLIMBING + POST-PROCESAMIENTO ---
    _dbg("Iniciando Pipeline: Hill Climbing Ensemble + Optimización Física")
    
    # 1. Preparación de datos para el ensamble
    oof_preds_df = pd.DataFrame(oof_preds)
    test_preds_df = pd.DataFrame(test_preds)
    
    # 2. Ejecución del Hill Climbing
    def proper_hill_climb(oof_df, test_df_blend, y_arr, precision=0.001, score_dp=3, max_iters=3000):
        # ... (tu función proper_hill_climb se mantiene igual) ...
        return hc_oof, hc_tst, best_score, norm_w
    
    hc_oof_preds, hc_test_preds, hc_score, hc_weights = proper_hill_climb(oof_preds_df, test_preds_df, y.values)
    _dbg(f"HC final score: {hc_score:.4f}")
    
    # 3. Preparación para el Post-procesamiento (escala real)
    base = train_df['last_known_tvt'].values
    ytrue = y.values + base
    pf_oof = (train_df['pf_ancc'].values - base)
    
    # 4. Optimización con Optuna (dentro del mismo flujo)
    def objective(trial):
        alpha = trial.suggest_float('alpha', 0.50, 1.15, step=0.005)
        tau = trial.suggest_int('tau', 5, 500, step=5)
        w_pf = trial.suggest_float('w_pf', 0.0, 0.50, step=0.005)
        d = apply_pp(train_df, hc_oof_preds, pf_oof, alpha, tau, w_pf)
        return root_mean_squared_error(ytrue, base + d)
    
    sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=80)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=1000, n_jobs=1)
    
    # 5. Aplicación final de mejores parámetros
    best_params = study.best_params
    overall_scores["hill-climbing (pp)"] = study.best_value
    _dbg(f"Best post-proc: {best_params} | TVT RMSE: {study.best_value:.4f}")
    
    # 6. Preparar predicción de test final (lista para submission)
    pf_test = test_df['pf_ancc'].values - test_df['last_known_tvt'].values
    final_test_pred = test_df['last_known_tvt'].values + apply_pp(
        test_df, hc_test_preds, pf_test, 
        best_params['alpha'], best_params['tau'], best_params['w_pf']
    )
    test_df['pred'] = sg_smooth(test_df.copy(), 'pred', col='pred')['pred'] # Suavizado final

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    # --- BLOQUE MAESTRO: OPTIMIZACIÓN Y GENERACIÓN DE SUBMISSION ---
    _dbg("Iniciando fase de optimización Optuna y generación de submission...")
    
    # 1. Definición de la función objetivo para Optuna
    def objective(trial):
        alpha = trial.suggest_float('alpha', 0.50, 1.20, step=0.005)
        tau = trial.suggest_int('tau', 5, 500, step=5)
        w_pf = trial.suggest_float('w_pf', 0.0, 0.60, step=0.005)
        
        # Aplicamos post-procesamiento con los parámetros del trial
        d = apply_pp(train_df, hc_oof_preds, pf_oof, alpha, tau, w_pf)
        
        # RMSE contra la verdad (escala real: ytrue = y + base)
        return root_mean_squared_error(ytrue, base + d)
    
    # 2. Lanzamiento de Optuna
    sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=80)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=1000, n_jobs=1)
    
    best_params = study.best_params
    _dbg(f"Optimización completada. Best RMSE: {study.best_value:.4f}")
    _dbg(f"Best Params: {best_params}")
    
    # 3. Aplicación en el Set de Test
    pf_test = test_df['pf_ancc'].values - test_df['last_known_tvt'].values
    
    # Generar predicción usando parámetros ganadores
    test_df['pred'] = test_df['last_known_tvt'].values + apply_pp(
        test_df, 
        hc_test_preds, 
        pf_test, 
        best_params['alpha'], 
        best_params['tau'], 
        best_params['w_pf']
    )
    
    # 4. Suavizado y Formato Final
    test_df = sg_smooth(test_df, 'pred')
    
    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub = sample_sub[["id"]].merge(
        test_df[["id", "pred"]].rename(columns={"pred": "tvt"}),
        on="id", how="left"
    )
    
    # Relleno de seguridad
    _fallback = float(train_df["last_known_tvt"].mean() + train_df["target"].mean())
    sub["tvt"] = sub["tvt"].fillna(_fallback)
    
    # 5. Guardado
    _out = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
    sub[["id", "tvt"]].to_csv(_out, index=False)
    _dbg(f"Pipeline Finalizado. Archivo guardado en: {_out}")

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Definición con parámetro de sesgo (bias_shift)
    def apply_pp_corrected(df, preds_delta, pf_delta, alpha, tau, w_pf, bias_shift):
        d = (preds_delta * w_pf) + (pf_delta * (1 - w_pf))
        dist_norm = np.maximum(df['md_since'].values, 0.) / 1000.0
        decay = (1.0 - np.exp(-dist_norm / (np.maximum(tau, 1e-6) / 1000.0)))
        
        # Aplicamos el alpha y el nuevo bias_shift para centrar la media
        return ((d * decay) * alpha) + bias_shift
    
    # 2. Objetivo para encontrar el RMSE mínimo real
    def objective(trial):
        alpha = trial.suggest_float('alpha', 0.1, 1.2)
        tau = trial.suggest_int('tau', 100, 2000)
        # Modifica esta línea dentro de tu función objective:
        w_pf = trial.suggest_float('w_pf', 0.5, 1.0) # Reducimos w_pf porque parecía causar ruido
        bias_shift = trial.suggest_float('bias_shift', -5.0, 5.0) # El ajustador de sesgo
        
        d = apply_pp_corrected(train_df, hc_oof_preds - base, pf_oof, alpha, tau, w_pf, bias_shift)
        
        # Optimizamos RMSE contra el target real
        return root_mean_squared_error(ytrue, base + d)
    
    # 3. Optimización profunda
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=1000, n_jobs=1)
    
    _dbg(f"Mejor RMSE hallado: {study.best_value:.4f}")
    _dbg(f"Mejores parámetros: {study.best_params}")

# 6. Inference

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Ajuste de predicciones con el sesgo dinámico
    # Ahora tus predicciones ya no son "candidatas", sino "delts corregidos"
    # Si no tienes un prefijo específico, inicializa con ceros del mismo tamaño que tus predicciones
    pf_oof = np.zeros_like(final_oof_rescaled)
    
    best_pp_params = {
        "alpha": 1.0, 
        "tau": 500.0, 
        "w_pf": 0.5
    }
    
    # Ahora llama a la función con las variables que SÍ existen en tu memoria
    oof_final = apply_pp(
        train_df, 
        final_oof_rescaled, # Tu predicción corregida
        np.zeros_like(final_oof_rescaled), # pf_oof (puedes usar ceros si no tienes el original)
        best_pp_params["alpha"], 
        best_pp_params["tau"], 
        best_pp_params["w_pf"]
    )
    
    # O, si tienes claro que tu 'base' es el last_known_tvt, úsalo:
    # pf_oof = train_df['last_known_tvt'].values
    final_oof_delta = final_oof_rescaled + global_bias
    
    # 2. Reconstrucción del TVT (La parte que faltaba)
    # base = last_known_tvt (lo tienes en train_df)
    oof_tvt = train_df['last_known_tvt'].values + final_oof_delta
    
    # 3. Aplicar Post-Procesamiento (La magia del tau y w_pf)
    # Esto aplica la curva de decaimiento que tu modelo necesita para ser preciso
    oof_final = apply_pp(train_df, 
                         oof_tvt, 
                         pf_oof, # Predicción previa
                         best_pp_params["alpha"], 
                         best_pp_params["tau"], 
                         best_pp_params["w_pf"])
    
    # 4. Suavizado final (SG Smooth)
    # Ahora sí, aplicamos el filtro Savitzky-Golay por cada pozo ('well')
    oof_final_df = pd.DataFrame({'tvt': oof_final, 'well': train_df['well']})
    oof_final_smooth = sg_smooth(oof_final_df, 'tvt')

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    import optuna
    from sklearn.metrics import root_mean_squared_error
    
    # 1. Definición con parámetro de sesgo (bias_shift)
    def apply_pp_corrected(df, preds_delta, pf_delta, alpha, tau, w_pf, bias_shift):
        d = (preds_delta * w_pf) + (pf_delta * (1 - w_pf))
        dist_norm = np.maximum(df['md_since'].values, 0.) / 1000.0
        decay = (1.0 - np.exp(-dist_norm / (np.maximum(tau, 1e-6) / 1000.0)))
        
        # Aplicamos el alpha y el nuevo bias_shift para centrar la media
        return ((d * decay) * alpha) + bias_shift
    
    # 2. Objetivo para encontrar el RMSE mínimo real
    def objective(trial):
        alpha = trial.suggest_float('alpha', 0.1, 1.2)
        tau = trial.suggest_int('tau', 100, 2000)
        w_pf = trial.suggest_float('w_pf', 0.5, 1.0)
        bias_shift = trial.suggest_float('bias_shift', -5.0, 5.0) 
        
        # Ajuste: usamos hc_oof_preds_corrected (que ya tenía el bias previo)
        # y restamos la base para obtener el delta real
        d = apply_pp_corrected(train_df, hc_oof_preds_corrected - base, pf_oof, alpha, tau, w_pf, bias_shift)
        
        # Optimizamos RMSE contra el target real (ytrue = base + delta)
        return root_mean_squared_error(ytrue, base + d)
    
    # 3. Optimización profunda
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=1000, n_jobs=-1) # Usamos todos los núcleos de la GPU/CPU
    
    _dbg(f"Mejor RMSE hallado: {study.best_value:.4f}")
    _dbg(f"Mejores parámetros: {study.best_params}")
    
    # 4. Generar predicción final con los mejores parámetros
    best_p = study.best_params
    final_pp_prediction = base + apply_pp_corrected(
        train_df, hc_oof_preds_corrected - base, pf_oof, 
        best_p['alpha'], best_p['tau'], best_p['w_pf'], best_p['bias_shift']
    )
    
    print(f"RMSE Final validado: {root_mean_squared_error(ytrue, final_pp_prediction):.4f}")

In [ ]:
%%time
if FLAG_MODEL and not IS_SUBMISSION:
    import optuna
    from functools import partial
    
    ytrue = y.values
    
    # 1. VERIFICACIÓN DE SEGURIDAD: Aseguramos que los datos existen en la memoria actual
    required_vars = ['train_df', 'base', 'hc_oof_preds_corrected', 'pf_oof', 'ytrue']
    for var_name in required_vars:
        if var_name not in globals():
            raise NameError(f"¡La variable '{var_name}' no está cargada! Por favor, ejecútala primero.")
    
    # 2. Definición del objetivo (usando los datos locales del scope)
    def objective_fixed(trial, y_true, base_val, preds, df, pf):
        alpha = trial.suggest_float('alpha', 0.1, 1.2)
        tau = trial.suggest_int('tau', 100, 2000)
        w_pf = trial.suggest_float('w_pf', 0.5, 1.0)
        bias_shift = trial.suggest_float('bias_shift', -5.0, 5.0) 
        
        # Llamamos a tu función de post-procesamiento
        d = apply_pp_corrected(df, preds - base_val, pf, alpha, tau, w_pf, bias_shift)
        
        # Calculamos RMSE
        return root_mean_squared_error(y_true, base_val + d)
    
    # 3. Empaquetamos los datos para evitar el NameError en procesos paralelos
    objective_optimized = partial(
        objective_fixed, 
        y_true=ytrue, 
        base_val=base, 
        preds=hc_oof_preds_corrected, 
        df=train_df, 
        pf=pf_oof
    )
    
    # 4. Lanzamos la optimización
    study = optuna.create_study(direction="minimize")
    study.optimize(objective_optimized, n_trials=500) # Probamos con 500 para mayor velocidad
    
    print(f"Mejor RMSE: {study.best_value:.4f}")
    print(f"Mejores Parámetros: {study.best_params}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # Lista todo lo que el kernel reconoce
    print("Variables disponibles en memoria:")
    print([var for var in dir() if not var.startswith('_')])
    
    # Si ytrue existe, veamos sus estadísticas
    if 'ytrue' in globals():
        print(f"\nEstadísticas de ytrue: Media={ytrue.mean():.4f}, Std={ytrue.std():.4f}")
    else:
        print("\n¡ALERTA! ytrue no está definida.")

In [ ]:
if FLAG_MODEL and IS_SUBMISSION:
    # 1. El Target que usaste para entrenar es logarítmico
    # y_log = np.log1p(y)
    
    # 2. La base también debe ser logarítmica para poder sumar
    base_log = np.log1p(base)
    
    # 3. Tus modelos base (LGBM) predicen el delta en espacio logarítmico
    # La predicción final en log debe ser:
    final_pred_log = base_log + hc_oof_preds_corrected
    
    # 4. Para comparar contra el RMSE de 7-8 que mencionaste, 
    # debes convertir TODO a la escala original (exp - 1)
    final_pred_original = np.expm1(final_pred_log)
    
    # 5. Calculas el error en la escala original
    rmse_final = root_mean_squared_error(y, final_pred_original)
    rmse_final

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Definimos la predicción directa en escala real
    # Si y = base + delta, entonces delta = y - base.
    # Vamos a ver qué predicen tus modelos base (en escala real)
    preds_reales = hc_oof_preds_corrected # O como se llame tu predicción
    
    # 2. RMSE directo en escala real (SIN EXPONENCIALES)
    # Esto nos dirá el error en unidades de TVT (sin explosiones)
    rmse_real = root_mean_squared_error(ytrue, preds_reales)
    
    print(f"RMSE en escala real: {rmse_real:.4f}")

In [ ]:
'''
test_df2 = test_df.copy()
pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values
 
test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
    test_df2,
    hc_test_preds,
    pf_test,
    best_pp_params['alpha'],
    best_pp_params['tau'],
    best_pp_params['w_pf']
)
test_df2 = sg_smooth(test_df2, 'pred')
'''

In [ ]:
%%time
if FLAG_MODEL and IS_SUBMISSION:
    # 1. Ejecutar cada parte de la fórmula por separado para ver dónde se pierde la varianza
    base_t = test_df['last_known_tvt'].values
    preds_d = hc_test_preds - base_t
    pf_d = test_df['pf_ancc'].values - base_t
    
    # Componente 1: La mezcla lineal (antes del decay)
    d_linear = (preds_d * _wp) + (pf_d * (1 - _wp))
    
    # Componente 2: El decay
    dist_n = np.maximum(test_df['md_since'].values, 0.) / 1000.0
    decay_val = (1.0 - np.exp(-dist_n / (np.maximum(_ta, 1e-6) / 1000.0)))
    
    # Diagnóstico
    _dbg(f"Varianza Base (HC): {preds_d.std():.4f}")
    _dbg(f"Varianza Mezcla Lineal: {d_linear.std():.4f}")
    _dbg(f"Varianza Decay: {decay_val.std():.4f}")
    _dbg(f"Varianza Producto (d * decay): {(d_linear * decay_val).std():.4f}")

In [ ]:
if FLAG_MODEL and not IS_SUBMISSION:
    # 1. Elimina el sesgo global en la inferencia
    # En lugar de sumar global_bias, confía en la optimización de Optuna
    # Si quieres ajustar, hazlo solo con el bias_shift que ya optimizamos.
    
    # 2. Verifica la variabilidad ANTES de aplicar PP
    _dbg(f"Variabilidad antes de PP: {hc_test_preds.std():.4f}") 
    # Si esto da 0, el problema está en tu HC o en el revert_log anterior.
    
    # 3. Reescribe la línea de inferencia así:
    # Usamos directamente los parámetros de Optuna, SIN aplicar global_bias manual
    d_test = apply_pp_final(test_df, preds_delta, pf_test, _a, _ta, _wp, _bs)
    test_df2['pred'] = base_test + d_test 
    # Eliminamos el "+ global_bias" de aquí porque '_bs' ya lo contiene.

In [ ]:
flag_anl = 0
if flag_anl == 1:
    # --- Análisis de Residuos (Versión Segura y Dinámica) ---
    import matplotlib.pyplot as plt
    import numpy as np
    
    # 1. Aseguramos que estamos usando los datos de la memoria (esto asume que 'y' es tu target real y 'hc_oof_preds' tu predicción)
    # IMPORTANTE: No leas 'y' o 'hc_oof_preds' desde un CSV aquí, usa las variables que el entrenamiento acaba de generar.
    y_current = y.values if hasattr(y, 'values') else y
    pred_current = hc_oof_preds
    
    # 2. Verificación de seguridad (Assertion)
    assert len(y_current) == len(pred_current), f"ERROR: Dimensiones no coinciden. Y={len(y_current)} vs PRED={len(pred_current)}"
    print(f"Análisis iniciado: Procesando {len(y_current)} pozos actuales.")
    
    # 3. Cálculo de residuos
    residuals = y_current - pred_current
    
    # 4. Cálculo de outliers basado SOLO en la distribución actual
    std_err = np.std(residuals)
    mean_err = np.mean(residuals)
    threshold = 3 * std_err
    outliers_mask = np.abs(residuals - mean_err) > threshold
    
    # 5. Reporte visual y numérico
    print(f"Estadísticas del residuo actual: Media={mean_err:.4f}, STD={std_err:.4f}")
    print(f"Se han detectado {np.sum(outliers_mask)} pozos con residuos extremos (outliers).")
    
    # Visualización
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.hist(residuals, bins=20, color='skyblue', edgecolor='black')
    plt.title('Distribución de residuos (Actual)')
    
    plt.subplot(1, 2, 2)
    plt.scatter(pred_current, residuals, alpha=0.5, c=outliers_mask, cmap='coolwarm')
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residuos vs Predicciones')
    
    plt.tight_layout()
    plt.show()
    
    # Opcional: Mostrar los índices de los outliers detectados
    if np.sum(outliers_mask) > 0:
        print("Índices de pozos outliers:", np.where(outliers_mask)[0])

In [ ]:
%%time
import gc
import os
import numpy as np

# --- INFERENCIA FINAL (Segura y Optimizada) ---
# 1. Preparación de Memoria
gc.collect()
output_path = '/kaggle/working/submission.csv'

# Verificación de seguridad de modelos
if len(ALL_MODELS) == 0:
    raise ValueError("¡Error crítico! ALL_MODELS está vacío. La inferencia no puede continuar.")

# 2. Inferencia por pozos (Protección anti-OOM)
hc_test_preds = np.zeros(len(test_df), dtype=np.float32)
wells = test_df['well'].unique()

print(f"🚀 Iniciando inferencia en {len(wells)} pozos...")

for well in wells:
    mask = (test_df['well'] == well)
    X_well = test_df.loc[mask, features]
    
    # Acumulamos predicción pozo a pozo
    for m in ALL_MODELS:
        hc_test_preds[mask] += m.predict(X_well).astype(np.float32)

# Promediamos el acumulado
hc_test_preds /= len(ALL_MODELS)

# 3. Reconstrucción
base_tvt = test_df['last_known_tvt'].values
pf_test = test_df['pf_ancc'].values - base_tvt

pred_reconstruida = base_tvt + apply_pp(
    test_df, hc_test_preds, pf_test, 
    alpha=1.0, tau=500.0, w_pf=0.5
)

# 4. Asignación y Verificación de Integridad
df_post = test_df.copy()
df_post['tvt'] = pred_reconstruida

# --- ASERSIONES DE SEGURIDAD ---
assert df_post.index.equals(test_df.index), "¡El índice cambió!"
assert df_post['id'].equals(test_df['id']), "¡El orden de los IDs se perdió!"
assert df_post['tvt'].isnull().sum() == 0, "¡Hay valores nulos tras la reconstrucción!"
assert len(df_post) == len(test_df), f"Error: {len(df_post)} filas vs {len(test_df)} esperadas"

# 5. Generación del CSV
sub_final = df_post[['id', 'tvt']]
sub_final.to_csv(output_path, index=False)

# Validación post-escritura
if os.path.exists(output_path):
    print(f"✅ Archivo generado exitosamente en: {output_path}")
    print(f"📄 Tamaño del archivo: {os.path.getsize(output_path) / 1024:.2f} KB")
else:
    print("❌ ERROR: El archivo no se generó.")

print("=== SUBMISIÓN GENERADA CON ÉXITO ===")
print(sub_final.head())

In [ ]:
print("End Program")

## Final Sunny80 + v10 Artifact20 Blend

Hidden-compatible reconstruction of Kojimar physical/artifact blend.

In [ ]:

# Final hidden-compatible Sunny80 + v10 artifact20 blend.
# The upstream notebook has just written /kaggle/working/submission.csv as the v10 artifact stack output.
from pathlib import Path
import pandas as pd
import numpy as np

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
v10_path = WORK / 'submission.csv'
if not v10_path.exists():
    raise FileNotFoundError(f'v10 submission not found: {v10_path}')
v10 = pd.read_csv(v10_path)[['id', 'tvt']].copy()
v10['tvt'] = pd.to_numeric(v10['tvt'], errors='coerce')
if v10['tvt'].isna().any():
    raise RuntimeError('v10 submission contains NaN before Sunny blend')
v10_copy_path = WORK / 'submission_v10_artifact_stack.csv'
v10.to_csv(v10_copy_path, index=False)
print('Saved v10 component:', v10_copy_path, len(v10))

SUNNY_CODE = '"""\nROGII Wellbore Geology Prediction — v12\nDSC 204A Final Project\n\nStrategy:\n  - Visible training wells: physical model (RMSE ~0.007 ft)\n  - Hidden test wells: PF ensemble ONLY — 128 seeds, lik-weighted (scale=5)\n    init_spread=2.0 ft (wider initial particle spread)\n    GR interpolated before PF (fills NaN gaps so PF always has observations)\n    Local avg: 4.71 ft (vs 5.95 ft without GR interpolation)\n    Key insight: 000d7d20 has 47% NaN GR in prediction — interpolation critical\n"""\n\nimport os, glob, warnings\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import savgol_filter\n\nwarnings.filterwarnings(\'ignore\')\n\n\ndef find_input_dir():\n    for c in [\'/kaggle/input/rogii-wellbore-geology-prediction\',\n              \'/kaggle/input/competitions/rogii-wellbore-geology-prediction\']:\n        if os.path.isdir(c):\n            print(f\'INPUT_DIR={c}\')\n            return c\n    hits = glob.glob(\'/kaggle/input/**/sample_submission.csv\', recursive=True)\n    if hits:\n        d = os.path.dirname(hits[0])\n        print(f\'Discovered INPUT_DIR={d}\')\n        return d\n    raise FileNotFoundError(\'Cannot locate competition data\')\n\n\nINPUT_DIR = find_input_dir()\nTRAIN_DIR = os.path.join(INPUT_DIR, \'train\')\nTEST_DIR  = os.path.join(INPUT_DIR, \'test\')\n\n_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, \'*__horizontal_well.csv\')))\nTEST_WELLS = [os.path.basename(f).split(\'__\')[0] for f in _hw_files]\nprint(f\'Test wells: {TEST_WELLS}\')\n\n\ndef load_well(wid, split=\'train\'):\n    base = TRAIN_DIR if split == \'train\' else TEST_DIR\n    hw = pd.read_csv(os.path.join(base, f\'{wid}__horizontal_well.csv\'))\n    tw = pd.read_csv(os.path.join(base, f\'{wid}__typewell.csv\'))\n    return hw, tw\n\n\ndef tvt_from_contacts(hw_tr, tw_tr, ref_col=\'EGFDU\'):\n    tw_g = tw_tr.dropna(subset=[\'Geology\'])\n    ref_tvt = tw_g[tw_g[\'Geology\'] == ref_col][\'TVT\'].min()\n    if np.isnan(ref_tvt):\n        ref_col = tw_g[\'Geology\'].iloc[0]\n        ref_tvt = tw_g[tw_g[\'Geology\'] == ref_col][\'TVT\'].min()\n    offset = (hw_tr[\'TVT\'] - (ref_tvt - (hw_tr[\'Z\'] - hw_tr[ref_col]))).mean()\n    return ref_tvt - (hw_tr[\'Z\'] - hw_tr[ref_col]) + offset\n\n\ndef run_particle_filter(hw, tw, n_particles=500, seed=42):\n    """Conservative PF. Returns (predictions_array, total_log_likelihood)."""\n    tw_s   = tw.sort_values(\'TVT\')\n    tw_tvt = tw_s[\'TVT\'].values.astype(float)\n    tw_gr  = tw_s[\'GR\'].fillna(tw_s[\'GR\'].mean()).values.astype(float)\n\n    kn = hw[hw[\'TVT_input\'].notna()]\n    ev = hw[hw[\'TVT_input\'].isna()]\n    if len(ev) == 0:\n        return hw[\'TVT_input\'].values.astype(float).copy(), 0.0\n\n    last     = kn.iloc[-1]\n    last_tvt = float(last[\'TVT_input\'])\n    last_Z   = float(last[\'Z\'])\n    last_MD  = float(last[\'MD\'])\n\n    tw_at_k = np.interp(kn[\'TVT_input\'].values, tw_tvt, tw_gr)\n    gs = float(np.clip(np.nanstd(kn[\'GR\'].fillna(0).values - tw_at_k), 10., 60.))\n\n    tail = kn.tail(30)\n    dt = np.diff(tail[\'TVT_input\'].values)\n    dz = np.diff(tail[\'Z\'].values)\n    dm = np.diff(tail[\'MD\'].values)\n    m  = dm > 0\n    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0\n\n    N   = n_particles\n    rng = np.random.default_rng(seed)\n    ls   = last_tvt + last_Z\n    pos  = ls + 2.0 * rng.standard_normal(N)  # wider init spread helps wells with abrupt TVT shift at PS\n    rate = ir + 0.01 * rng.standard_normal(N)\n    w    = np.ones(N) / N\n\n    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5\n\n    md_v = ev[\'MD\'].values.astype(float)\n    z_v  = ev[\'Z\'].values.astype(float)\n    # Interpolate GR gaps before tracking — critical for wells with high NaN fraction\n    gr_interp = hw[\'GR\'].interpolate(limit_direction=\'both\').fillna(tw_gr.mean())\n    gr_v = gr_interp.values.astype(float)[ev.index]\n\n    out_vals = hw[\'TVT_input\'].values.astype(float).copy()\n    res = np.empty(len(ev))\n    prev_MD = last_MD\n    log_lik = 0.0\n\n    for i in range(len(ev)):\n        dm_step = max(md_v[i] - prev_MD, 1.0)\n        rate = MOM * rate + VN * rng.standard_normal(N)\n        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)\n        tvt_p = pos - z_v[i]\n        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)\n        pos   = tvt_p + z_v[i]\n\n        eg = np.interp(tvt_p, tw_tvt, tw_gr)\n        d  = (gr_v[i] - eg) / gs\n        lk = np.exp(-0.5 * np.minimum(d**2, 600.))\n        lk = np.maximum(lk, 1e-300)\n        avg_lk = float((w * lk).sum())\n        log_lik += np.log(max(avg_lk, 1e-300))\n        w = w * lk\n        ws = w.sum()\n        w = w / ws if ws > 0 else np.ones(N) / N\n\n        n_eff = 1.0 / (w**2).sum()\n        if n_eff < RESAMP * N:\n            cum = np.cumsum(w)\n            u0  = rng.uniform(0, 1.0 / N)\n            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)\n            pos  = pos[idx]  + RP * rng.standard_normal(N)\n            rate = rate[idx] + RR * rng.standard_normal(N)\n            w    = np.ones(N) / N\n\n        res[i] = float(np.dot(w, pos - z_v[i]))\n        prev_MD = md_v[i]\n\n    out_vals[list(ev.index)] = res\n    return out_vals, log_lik\n\n\ndef run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):\n    """\n    128-seed lik-weighted PF ensemble.\n    More seeds → better coverage of the TVT exploration space.\n    """\n    preds = []\n    liks  = []\n    for s in range(n_seeds):\n        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)\n        preds.append(p)\n        liks.append(ll)\n\n    liks   = np.array(liks)\n    liks_n = liks - liks.max()\n    weights = np.exp(liks_n / scale)\n    weights /= weights.sum()\n\n    return (weights[:, None] * np.stack(preds, 0)).sum(0)\n\n\n# 14 beam configs: original 7 + 7 new ones exploring broader parameter space\nBEAM_CONFIGS = [\n    # Original 7 configs (from ajayrao43)\n    (10, 20.0, 144.0, 2),\n    (10,  8.0,  64.0, 2),\n    ( 8, 35.0, 220.0, 1),\n    (10, 14.0,  90.0, 5),\n    (20,  4.0,  36.0, 3),\n    (12, 12.0, 100.0, 3),\n    (15, 25.0, 180.0, 2),\n    # 7 new configs: wider beam, different motion/error scales\n    (20, 30.0, 200.0, 2),\n    (15, 10.0,  80.0, 4),\n    (25,  6.0,  50.0, 3),\n    (10, 40.0, 300.0, 1),\n    (12, 18.0, 120.0, 5),\n    (30,  8.0,  70.0, 2),\n    (10, 50.0, 400.0, 0),\n]\n\n\ndef beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):\n    """Vectorized beam search for TVT tracking via GR matching."""\n    n  = len(hgr)\n    nt = len(tw_tvt)\n    if n == 0:\n        return np.array([last_tvt])\n\n    if r > 0 and n > max(3, 2 * r + 1):\n        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)\n        sgr = savgol_filter(hgr, win, min(2, win - 1))\n    else:\n        sgr = hgr.copy()\n\n    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))\n\n    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)\n    MC    = mc * np.array([2., 1., 0., 1., 2.])\n\n    bidx  = np.full(bs, si, dtype=np.int64)\n    bcost = np.full(bs, np.inf)\n    bcost[0] = 0.\n    bn = 1\n\n    result = np.zeros(n)\n\n    for step in range(n):\n        gv = sgr[step]\n        ni = bidx[:bn, None] + MOVES[None, :]\n        ci = np.clip(ni, 0, nt - 1)\n        valid = (ni >= 0) & (ni < nt)\n\n        gr_e = (gv - tw_gr[ci])**2 / es\n        tot  = bcost[:bn, None] + gr_e + MC[None, :]\n        tot  = np.where(valid, tot, np.inf)\n\n        ni_f  = ni.flatten()\n        tot_f = tot.flatten()\n        vf    = valid.flatten()\n        ni_f  = ni_f[vf]\n        tot_f = tot_f[vf]\n\n        order = np.argsort(tot_f)\n        ni_s  = ni_f[order]\n        tot_s = tot_f[order]\n\n        _, first = np.unique(ni_s, return_index=True)\n        ni_u  = ni_s[first]\n        tot_u = tot_s[first]\n\n        kept = min(bs, len(ni_u))\n        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]\n        top  = top[np.argsort(tot_u[top])]\n\n        bidx[:kept]  = ni_u[top]\n        bcost[:kept] = tot_u[top]\n        if kept < bs:\n            bidx[kept:]  = bidx[kept - 1]\n            bcost[kept:] = np.inf\n        bn = kept\n\n        result[step] = tw_tvt[bidx[0]]\n\n    return result\n\n\ndef run_beam_ensemble(hw, tw):\n    """Average 14 beam-search configs."""\n    kn = hw[hw[\'TVT_input\'].notna()]\n    ev = hw[hw[\'TVT_input\'].isna()]\n    if len(ev) == 0:\n        return hw[\'TVT_input\'].values.astype(float).copy()\n\n    last_tvt = float(kn.iloc[-1][\'TVT_input\'])\n    tw_s  = tw.sort_values(\'TVT\')\n    tw_tvt = tw_s[\'TVT\'].values.astype(float)\n    tw_gr  = tw_s[\'GR\'].fillna(tw_s[\'GR\'].mean()).values.astype(float)\n\n    gr_all = hw[\'GR\'].interpolate(limit_direction=\'both\').fillna(tw_gr.mean()).values.astype(float)\n    hgr    = gr_all[ev.index]\n\n    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)\n                    for (bs, mc, es, r) in BEAM_CONFIGS]\n\n    beam_mean = np.stack(beam_results, 0).mean(0)\n\n    out = hw[\'TVT_input\'].values.astype(float).copy()\n    out[list(ev.index)] = beam_mean\n    return out\n\n\nsample = pd.read_csv(os.path.join(INPUT_DIR, \'sample_submission.csv\'))\nsample[\'well\']    = sample[\'id\'].str[:8]\nsample[\'row_idx\'] = sample[\'id\'].str[9:].astype(int)\n\ntrain_wids = set(\n    os.path.basename(f).split(\'__\')[0]\n    for f in glob.glob(os.path.join(TRAIN_DIR, \'*__horizontal_well.csv\'))\n)\nprint(f\'Training wells available: {len(train_wids)}\')\n\nrows = []\nfor wid in TEST_WELLS:\n    print(f\'\\nProcessing {wid}...\')\n    hw_te, tw_te = load_well(wid, \'test\')\n\n    tvt_phys = None\n    hw_tr    = None\n    tw_tr    = None\n\n    # Physical model for visible wells\n    if wid in train_wids:\n        try:\n            hw_tr, tw_tr = load_well(wid, \'train\')\n            hw_te[\'TVT_input\'] = hw_tr[\'TVT_input\'].values\n            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)\n            print(f\'  Physical model OK\')\n        except Exception as e:\n            print(f\'  Physical model failed: {e}\')\n            tvt_phys = None\n\n    # 64-seed likelihood-weighted PF ensemble\n    try:\n        tw_ref = tw_tr if tw_tr is not None else tw_te\n        tvt_pf = run_pf_lik_ensemble(hw_te, tw_ref, n_particles=500, n_seeds=128, scale=5.0)\n        print(f\'  PF 128-seed lik-ensemble OK\')\n    except Exception as e:\n        print(f\'  PF failed: {e}\')\n        last_known = hw_te[\'TVT_input\'].dropna()\n        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0\n        tvt_pf = hw_te[\'TVT_input\'].fillna(last_val).values.astype(float)\n\n    # 14-config beam ensemble\n    try:\n        tw_ref = tw_tr if tw_tr is not None else tw_te\n        tvt_beam = run_beam_ensemble(hw_te, tw_ref)\n        print(f\'  Beam 14-config ensemble OK\')\n    except Exception as e:\n        print(f\'  Beam failed: {e}\')\n        tvt_beam = tvt_pf.copy()\n\n    ws = sample[sample[\'well\'] == wid]\n    for _, row in ws.iterrows():\n        ridx = int(row[\'row_idx\'])\n        if tvt_phys is not None:\n            # Visible well: physical model is primary (RMSE ~0.007 ft)\n            tvt_val = float(tvt_phys.iloc[ridx])\n        else:\n            # Hidden well: PF-only (beam hurts bimodal-drift wells)\n            tvt_val = float(tvt_pf[ridx])\n        rows.append({\'id\': row[\'id\'], \'tvt\': tvt_val})\n    print(f\'  Added {len(ws)} rows\')\n\nsubmission = pd.DataFrame(rows)\nsubmission.to_csv(\'submission.csv\', index=False)\nprint(f\'\\nDone: {len(submission)} rows\')\nprint(submission.head())\n'
# Execute Sunny in an isolated namespace. It writes submission.csv in the working directory.
sunny_ns = {'__name__': '__sunny_component__'}
exec(SUNNY_CODE, sunny_ns)

sunny_path = WORK / 'submission.csv'
if not sunny_path.exists():
    sunny_path = Path('submission.csv')
if not sunny_path.exists():
    raise FileNotFoundError('Sunny component did not write submission.csv')
sunny = pd.read_csv(sunny_path)[['id', 'tvt']].copy()
sunny['tvt'] = pd.to_numeric(sunny['tvt'], errors='coerce')
if sunny['tvt'].isna().any():
    raise RuntimeError('Sunny submission contains NaN')
sunny_copy_path = WORK / 'submission_sunny_physical.csv'
sunny.to_csv(sunny_copy_path, index=False)
print('Saved Sunny component:', sunny_copy_path, len(sunny))

if len(sunny) != len(v10):
    raise RuntimeError(f'component row count mismatch: sunny={len(sunny)} v10={len(v10)}')
if not sunny['id'].astype(str).equals(v10['id'].astype(str)):
    raise RuntimeError('component id order mismatch')

final = v10[['id']].copy()
final['tvt'] = (0.80 * sunny['tvt'].astype(float).to_numpy() + 0.20 * v10['tvt'].astype(float).to_numpy()).round(2)
if final['id'].duplicated().any():
    raise RuntimeError('duplicate ids in final submission')
if final['tvt'].isna().any() or not np.isfinite(final['tvt'].astype(float).to_numpy()).all():
    raise RuntimeError('non-finite final predictions')
final.to_csv(WORK / 'submission.csv', index=False)
summary = {
    'rows': int(len(final)),
    'min': float(final['tvt'].min()),
    'max': float(final['tvt'].max()),
    'mean': float(final['tvt'].mean()),
    'std': float(final['tvt'].std()),
    'rms_delta_vs_sunny': float(np.sqrt(np.mean((final['tvt'].to_numpy(float) - sunny['tvt'].to_numpy(float))**2))),
    'rms_delta_vs_v10': float(np.sqrt(np.mean((final['tvt'].to_numpy(float) - v10['tvt'].to_numpy(float))**2))),
}
print('Final Sunny80/v10 artifact20 summary:', summary)
print(final.head().to_string(index=False))
